# Eksperimen LoRA vs Full Fine-Tuning pada Sentence-BERT (STS) — Google Colab Pro (v4, Opsi 2: seluruh rangkaian wajib)

**Cara pakai (singkat):** (1) Runtime → *Change runtime type* → **L4 GPU** (satu jenis GPU untuk seluruh run utama). (2) Jalankan sel **berurutan dari atas** (atau *Run all*). (3) Jika sesi putus, buka notebook lagi lalu **Runtime → Run all**: run yang selesai **dilewati**, run yang terputus **dilanjutkan dari epoch terakhir**. (4) **Jangan ubah** `ROOT`, seed, grid LR, `DELTA`, atau konfigurasi lain setelah berjalan. (5) Folder Drive v3 **dibiarkan** (run v3 yang konfigurasinya identik otomatis dipakai ulang; tabel v4 ditulis ke folder baru `tables_v4/` dan `figures/`).

**Tidak ada bagian opsional.** Seluruh rangkaian dijalankan: pencarian LR **per rank**, screening 30 konfigurasi dengan *fallback* otomatis bila *collapse*, konfirmasi **semua strategi × semua metode × 5 seed**, **10 seed** untuk kontras utama, ablasi (rsLoRA, LoRA q,v-saja), benchmark GPU, evaluasi test sekali, statistik RQ1–RQ3, laporan komputasi, serta **tabel, grafik, algoritma, dan rumus** siap naskah.

**Perubahan terhadap v3 (alasan: temuan *collapse* LoRA pada validasi, bukan pada test)**
0. **v4.6 (analisis ulang, tanpa GPU):** kriteria *collapse* diperluas dengan (c) *collapse sementara*: Spearman validasi < 0,50 pada epoch mana pun (mis. 0,425 di epoch 2 lalu pulih ke 0,80, atau 0,022 lalu naik pelan ke 0,54, yang lolos dari kriteria v4.5 karena puncak ≥ 0,50 dan tidak ada penurunan akhir). Kriteria hanya memakai data VALIDASI, tetapi diperluas SETELAH hasil v4.5 dilihat, sehingga dilaporkan sebagai analisis dengan dua kriteria (hasil v4.5 = analisis sensitivitas). Keluaran ditulis ke `tables_v4b/`, `figures_v4b/`, `luaran_naskah_v4b.zip` agar tidak menimpa hasil v4.5. Jalankan *Run all*: tahap pelatihan dilewati, evaluasi memakai cache.
1. Setiap run punya penanda `diverged` (Spearman validasi terbaik < 0,50, atau jatuh > 0,10 dari puncak ke epoch terakhir). Run *collapse* dikeluarkan dari rerata dan dilaporkan sebagai **laju divergensi**. (v4.5: checkpoint `state.pt` lama yang terpotong/rusak di Drive (sisa tulis yang terputus saat kuota habis) kini dideteksi, dihapus otomatis, dan run diulang dari awal, bukan menghentikan sel dengan `EOFError`. v4.4: state optimizer & bobot sementara ditulis ke disk LOKAL VM dan hanya hasil akhir yang disalin ke Drive, untuk menghindari kuota operasi/bandwidth Drive; evaluasi melewati run collapse. v4.3: pembacaan Drive kini diulang bila gagal sesaat dan ada verifikasi kelengkapan run sebelum evaluasi. v4.1: collapse kini juga terdeteksi bila Spearman validasi jatuh > 0,10 dari puncaknya sampai epoch terakhir.)
2. LR LoRA dicari **per rank** (grid lebih aman 1e-4…1e-3), bukan hanya pada r = 8.
3. Screening otomatis menurunkan LR satu langkah bila sebuah konfigurasi *collapse* (dan **berhenti dengan peringatan** bila grid habis).
4. Perbandingan RQ1 dilakukan pada **strategi yang sama** (matched; S0/S1/S2) dan sebagai analisis kedua **terbaik vs terbaik** dengan 10 seed.
5. **Tahap 4b (baru di v4.2): uji stabilitas LR** pada 2 seed tambahan (43, 44) untuk SEMUA 30 konfigurasi; bila ada yang *collapse*, LR diturunkan otomatis dan diuji ulang sebelum Tahap 5.
6. Konfirmasi memakai **semua** strategi (diperlukan untuk interaksi metode × strategi, RQ2) dan koreksi Holm **per famili uji**.
7. Ablasi, benchmark, tabel, grafik, algoritma, rumus kini bagian wajib.

**Tahapan:** 0 Setup → Pengaturan & kunci GPU → Preflight → 1 Data → 2 Pilot & estimasi → 3 Pencarian LR per rank → 4 Screening + gerbang *collapse* → 4b Uji stabilitas LR (2 seed tambahan) → 5 Konfirmasi (5 seed, semua konfigurasi) → 5b Seed tambahan (10 seed) → 6 Ablasi → 7 Benchmark → 7b Verifikasi kelengkapan → 8 Evaluasi test (EN & transfer ID) → 9 Statistik RQ1–RQ3 → 10 Laporan komputasi → 11 Tabel & grafik naskah → 12 Algoritma & rumus → Paket luaran.

Setiap sel diawali blok **Keterangan**: *Library/plugin*, *Algoritma*, *Model*, *Alasan*.

## Tahap 0 — Setup
**Keterangan**
- *Library/plugin:* `google.colab.drive` (penyimpanan permanen); `torch` (GPU); `transformers` (Wolf dkk., 2020); `peft` (LoRA resmi Hugging Face); `datasets`; `scipy`; `pandas`/`numpy`; `pynvml` (bawaan Colab, membaca utilisasi/daya GPU) dan `psutil` (RAM).
- *Algoritma:* — (persiapan lingkungan).
- *Model:* belum dimuat.
- *Alasan:* disk Colab (`/content`) hilang saat sesi berakhir; semua checkpoint dan log harus di Drive agar proses tidak diulang. Versi library dan spesifikasi mesin dicatat agar eksperimen dapat direproduksi dan biaya komputasinya dapat dilaporkan.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
ROOT = '/content/drive/MyDrive/sts_lora'          # semua keluaran ada di sini (JANGAN diubah setelah mulai)
for d in ['code', 'runs', 'stages', 'tables_v4b', 'figures_v4b', 'logs', 'bench']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U transformers peft datasets accelerate
!pip uninstall -y -q torchao          # torchao bawaan Colab (0.10) lebih lama dari yang diminta PEFT; tidak dipakai
import torch, transformers, peft, datasets
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| datasets', datasets.__version__)
!pip freeze > {ROOT}/stages/requirements_freeze.txt
assert torch.cuda.is_available(), 'GPU tidak aktif: Runtime > Change runtime type > GPU'

**Keterangan — pustaka eksperimen (`sts_lora_lib.py`)**
- *Library/plugin:* `torch`, `transformers.AutoModel/AutoTokenizer`, `peft.LoraConfig/get_peft_model`, `scipy.stats`, `pynvml`, `psutil`.
- *Algoritma:* **Sentence-BERT** (encoder siamese + *mean pooling* + kosinus; Reimers & Gurevych, 2019); **LoRA** (ΔW = BA pada semua lapisan linear, `lora_alpha = 2r`, dropout 0,1; Hu dkk., 2022) dengan opsi **rsLoRA** (Kalajdzievski, 2023) dan target q,v-saja; **AdamW** (Loshchilov & Hutter, 2019) + *scheduler* linier + *warmup*; strategi **S0** (MSE kosinus), **S1** (Layer-wise LR Decay 0,9; Howard & Ruder, 2018), **S2** (**CoSENT**; Su, 2022); presisi campuran bf16/fp16 otomatis; **pemantau sumber daya** (thread sampling tiap 5 detik; energi = integral daya terhadap waktu).
- *Model:* dipilih lewat konfigurasi (`xlm-roberta-base`, `roberta-base`).
- *Alasan:* loop pelatihan ditulis sendiri (bukan `Trainer`) agar checkpoint per epoch (bobot terlatih, optimizer, scheduler, RNG, ringkasan sumber daya) dan pengukuran memori/waktu/energi per run identik untuk semua metode — syarat perbandingan yang adil. Sel ini hanya menulis berkas ke Drive.

In [ ]:
%%writefile /content/drive/MyDrive/sts_lora/code/sts_lora_lib.py
"""sts_lora_lib.py - pustaka eksperimen: LoRA vs full fine-tuning pada Sentence-BERT (STS).

Prinsip desain
--------------
* Satu "run" = (model, metode, rank, strategi, LR, seed). ID run deterministik (hash konfigurasi).
* Checkpoint per epoch (bobot terlatih + optimizer + scheduler + scaler + RNG) ditulis atomik ke
  folder run. Jika sesi Colab putus, memanggil run_train() lagi MELANJUTKAN dari epoch terakhir;
  jika sudah selesai, dilewati (tidak diulang).
* Satu file result.json per run (metrik validasi, parameter terlatih, memori puncak, waktu, dst.).
* Evaluasi test / transfer dilakukan terpisah (eval_run) dan prediksinya disimpan (.npy) untuk
  bootstrap berpasangan.
"""
from __future__ import annotations

import datetime
import hashlib
import json
import pickle
import os
import platform
import random
import re
import shutil
import threading
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr, rankdata, spearmanr
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

# ----------------------------------------------------------------------------- konfigurasi
DIVERGE_SPEARMAN = 0.50   # run dianggap collapse bila Spearman validasi TERBAIK < 0,50 (di bawah baseline tanpa fine-tuning)
STRATEGIES = {
    "S0": "cosine-MSE loss, LR seragam, scheduler linear+warmup (baseline)",
    "S1": "cosine-MSE loss + Layer-wise LR Decay (LLRD)",
    "S2": "CoSENT loss, LR seragam, scheduler linear+warmup",
}


@dataclass
class Config:
    model_name: str = "xlm-roberta-base"
    method: str = "full"          # "full" | "lora"
    rank: int = 0                 # hanya untuk lora
    strategy: str = "S0"          # S0 | S1 | S2
    lr: float = 2e-5
    seed: int = 42
    epochs: int = 4
    batch_size: int = 32
    max_len: int = 128
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    llrd_decay: float = 0.9
    lora_dropout: float = 0.1
    cosent_scale: float = 20.0
    train_data: str = "stsb_en"
    rslora: bool = False                  # OPSIONAL (ablasi): rank-stabilized LoRA
    lora_targets: str = "all-linear"      # OPSIONAL (ablasi): "all-linear" | "qv" (hanya query & value)

    def run_id(self) -> str:
        d = asdict(self)
        # field opsional hanya masuk hash bila bukan default -> ID run lama tetap valid
        if not d["rslora"]:
            d.pop("rslora")
        if d["lora_targets"] == "all-linear":
            d.pop("lora_targets")
        name = d["model_name"].replace("/", "-")
        tag = "full" if self.method == "full" else f"lora-r{self.rank}"
        if self.method == "lora" and self.rslora:
            tag += "-rs"
        if self.method == "lora" and self.lora_targets != "all-linear":
            tag += "-" + self.lora_targets
        h = hashlib.sha1(json.dumps(d, sort_keys=True).encode()).hexdigest()[:8]
        return f"{name}_{tag}_{self.strategy}_lr{self.lr:g}_s{self.seed}_{h}"


# ----------------------------------------------------------------------------- data
def load_data(name: str, split: str) -> pd.DataFrame:
    """Kembalikan DataFrame [sentence1, sentence2, label] dengan label dinormalisasi ke 0-1.

    ID dataset (diverifikasi pada Hugging Face):
      stsb_en     : sentence-transformers/stsb      (train/validation/test; kolom score 0-1)
      semrel_en   : SemRel/SemRel2024, config eng   (train/dev/test; kolom label)
      semrel_id   : SemRel/SemRel2024, config ind   (dev/test saja)
      stsb_id     : LazarusNLP/stsb_mt_id           (validation/test; skor asli di kolom 'correlation', 0-5)
    """
    from datasets import load_dataset

    if name == "stsb_en":
        ds = load_dataset("sentence-transformers/stsb", split=split)
        df = pd.DataFrame({"sentence1": ds["sentence1"], "sentence2": ds["sentence2"],
                           "label": np.asarray(ds["score"], dtype=float)})
    elif name in ("semrel_en", "semrel_id"):
        cfg = "eng" if name == "semrel_en" else "ind"
        ds = load_dataset("SemRel/SemRel2024", cfg, split=split)
        df = pd.DataFrame({"sentence1": ds["sentence1"], "sentence2": ds["sentence2"],
                           "label": np.asarray(ds["label"], dtype=float)})
    elif name == "stsb_id":
        ds = load_dataset("LazarusNLP/stsb_mt_id", split=split)
        # PERINGATAN: kolom 'score' hanyalah nomor urut; skor kemiripan sebenarnya = 'correlation'.
        df = pd.DataFrame({"sentence1": ds["text_1"], "sentence2": ds["text_2"],
                           "label": np.asarray([float(x) for x in ds["correlation"]]) / 5.0})
    else:
        raise ValueError(name)
    df["label"] = df["label"].clip(0, 1)
    return df.reset_index(drop=True)


# ----------------------------------------------------------------------------- model
class SBERT(torch.nn.Module):
    """Siamese encoder + mean pooling (arsitektur Sentence-BERT, Reimers & Gurevych, 2019)."""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def embed(self, enc):
        out = self.backbone(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
        h = out.last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)


def _load_retry(path, map_location, tries=5, **kw):
    """torch.load dengan percobaan ulang: Google Drive (FUSE) kadang gagal sesaat (FileNotFoundError/OSError saat baca)."""
    for k in range(tries):
        try:
            return torch.load(path, map_location=map_location, **kw)
        except OSError as e:
            if k == tries - 1:
                raise
            print(f"  [Drive] gagal membaca {Path(path).name} ({e!r}); coba lagi {k + 1}/{tries - 1}")
            time.sleep(5 * (k + 1))


def build_model(cfg: Config):
    try:
        backbone = AutoModel.from_pretrained(cfg.model_name, add_pooling_layer=False)
    except TypeError:
        backbone = AutoModel.from_pretrained(cfg.model_name)
    if cfg.method == "lora":
        from peft import LoraConfig, get_peft_model

        targets = "all-linear" if cfg.lora_targets == "all-linear" else ["query", "value"]
        # standar: alpha = 2r (skala alpha/r = 2). rsLoRA: skala alpha/sqrt(r); alpha dikalibrasi 2*sqrt(8) agar skala sama dengan
        # LoRA standar pada r=8 (r>8 -> skala lebih kecil, r<8 -> lebih besar), sehingga ablasi menguji peran skala vs rank.
        alpha = 2 * (8 ** 0.5) if cfg.rslora else 2 * cfg.rank
        lc = LoraConfig(r=cfg.rank, lora_alpha=alpha, lora_dropout=cfg.lora_dropout,
                        target_modules=targets, bias="none", use_rslora=cfg.rslora)
        backbone = get_peft_model(backbone, lc)
    elif cfg.method != "full":
        raise ValueError(cfg.method)
    return SBERT(backbone)


def trainable_state(model: SBERT) -> dict:
    return {k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}


def count_params(model: SBERT):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    return tr, tot


# ----------------------------------------------------------------------------- loss & optimizer
def cosine_mse_loss(u, v, y):
    return F.mse_loss(F.cosine_similarity(u, v), y)


def cosent_loss(u, v, y, scale=20.0):
    """CoSENT (Su, 2022): log(1 + sum_{y_i>y_j} exp(scale*(cos_j - cos_i)))."""
    s = F.cosine_similarity(u, v) * scale
    diff = s[None, :] - s[:, None]                 # diff[i,j] = s_j - s_i
    mask = (y[:, None] > y[None, :]).float()       # pasangan i lebih mirip dari j
    diff = diff - (1 - mask) * 1e12
    diff = torch.cat([torch.zeros(1, device=diff.device), diff.flatten()])
    return torch.logsumexp(diff, 0)


def _layer_index(name: str, n_layers: int):
    m = re.search(r"encoder\.layer\.(\d+)\.", name)
    if m:
        return int(m.group(1))
    if "embeddings" in name:
        return -1
    return n_layers  # parameter lain (mis. pooler) diperlakukan seperti layer teratas


def build_optimizer(model: SBERT, cfg: Config):
    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    nd = ("bias", "LayerNorm.weight", "layer_norm")
    groups = []
    if cfg.strategy == "S1":
        n_layers = model.backbone.config.num_hidden_layers if hasattr(model.backbone, "config") \
            else model.backbone.base_model.model.config.num_hidden_layers
        buckets = {}
        for n, p in params:
            li = _layer_index(n, n_layers)
            wd = 0.0 if any(x in n for x in nd) else cfg.weight_decay
            # LR layer teratas = lr; tiap layer ke bawah dikalikan llrd_decay; embedding terendah
            lr = cfg.lr * cfg.llrd_decay ** (n_layers - 1 - li) if li < n_layers else cfg.lr
            buckets.setdefault((lr, wd), []).append(p)
        groups = [{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in buckets.items()]
    else:
        dec = [p for n, p in params if not any(x in n for x in nd)]
        nod = [p for n, p in params if any(x in n for x in nd)]
        groups = [{"params": dec, "lr": cfg.lr, "weight_decay": cfg.weight_decay},
                  {"params": nod, "lr": cfg.lr, "weight_decay": 0.0}]
        groups = [g for g in groups if g["params"]]
    return torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)


# ----------------------------------------------------------------------------- util
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _io_retry(fn, what="", tries=6, wait=30):
    """Ulangi operasi berkas bila Google Drive menolak sesaat (kuota operasi/bandwidth -> OSError 'Input/output error')."""
    for k in range(tries):
        try:
            return fn()
        except OSError as e:
            if k == tries - 1:
                raise
            print(f"  [Drive] {what} gagal ({e!r}); tunggu {wait * (k + 1)} dtk lalu coba lagi ({k + 1}/{tries - 1})")
            time.sleep(wait * (k + 1))


def atomic_save(obj, path: Path):
    tmp = Path(str(path) + ".tmp")

    def go():
        torch.save(obj, tmp)
        os.replace(tmp, path)

    _io_retry(go, f"menulis {Path(path).name}")


def atomic_json(obj, path: Path):
    tmp = Path(str(path) + ".tmp")

    def go():
        tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False))
        os.replace(tmp, path)

    _io_retry(go, f"menulis {Path(path).name}")


def copy_to_drive(src: Path, dst: Path):
    """Salin SATU kali berkas dari disk lokal VM ke Drive (atomik, diverifikasi ukuran, dengan retry)."""
    tmp = Path(str(dst) + ".tmp")

    def go():
        shutil.copyfile(src, tmp)
        if tmp.stat().st_size != Path(src).stat().st_size:
            raise OSError("ukuran berkas hasil salin tidak sama")
        os.replace(tmp, dst)

    _io_retry(go, f"menyalin {Path(dst).name} ke Drive")


def env_info(device):
    import peft
    import transformers

    return {"torch": torch.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__,
            "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"}


def pick_precision(device):
    if device.type != "cuda":
        return "fp32"
    # bf16 hanya native pada GPU Ampere ke atas (compute capability >= 8, mis. A100).
    # T4 (7.5) hanya mengemulasikan bf16 (lambat) -> gunakan fp16 + GradScaler.
    return "bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp16"


def _autocast(precision, device):
    if precision == "fp32":
        return torch.autocast(device_type="cpu", enabled=False)
    dt = torch.bfloat16 if precision == "bf16" else torch.float16
    return torch.autocast(device_type=device.type, dtype=dt)


def _batches(df, bs, order=None):
    idx = order if order is not None else np.arange(len(df))
    for i in range(0, len(idx), bs):
        sub = df.iloc[idx[i:i + bs]]
        yield sub["sentence1"].tolist(), sub["sentence2"].tolist(), \
            torch.tensor(sub["label"].values, dtype=torch.float32)


@torch.no_grad()
def predict(model, tok, df, cfg, device, precision, bs=128):
    model.eval()
    out = []
    for s1, s2, _ in _batches(df, bs):
        e1 = tok(s1, padding=True, truncation=True, max_length=cfg.max_len, return_tensors="pt").to(device)
        e2 = tok(s2, padding=True, truncation=True, max_length=cfg.max_len, return_tensors="pt").to(device)
        with _autocast(precision, device):
            u, v = model.embed(e1), model.embed(e2)
        out.append(F.cosine_similarity(u.float(), v.float()).cpu().numpy())
    return np.concatenate(out)


def metrics(pred, y):
    return {"spearman": float(spearmanr(pred, y)[0]), "pearson": float(pearsonr(pred, y)[0])}


# ----------------------------------------------------------------------------- log komputasi & sumber daya
_LOG = {"dir": None, "stage": "unset", "session": "nosession"}


def set_log_dir(root):
    """Folder log (JSONL peristiwa komputasi) di Drive."""
    _LOG["dir"] = Path(root) / "logs"
    _LOG["dir"].mkdir(parents=True, exist_ok=True)


def set_stage(name: str):
    _LOG["stage"] = name


def log_event(**kw):
    """Tambahkan satu baris JSON ke logs/compute_log.jsonl. Tidak pernah menggagalkan pelatihan."""
    if _LOG["dir"] is None:
        return
    try:
        rec = {"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
               "session": _LOG["session"], "stage": _LOG["stage"], **kw}
        with open(_LOG["dir"] / "compute_log.jsonl", "a") as f:
            f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
    except Exception:
        pass


def log_units(remaining: float, note: str = ""):
    """Catat sisa compute units SECARA MANUAL (baca dari panel resources Colab) -> konsumsi unit nyata."""
    log_event(event="units_remaining", remaining=float(remaining), note=note)


def log_session_start(root, extra=None):
    """Catat snapshot lingkungan sesi (GPU, driver, CPU, RAM, disk, versi library)."""
    import peft
    import transformers

    _LOG["session"] = uuid.uuid4().hex[:8]
    env = {"python": platform.python_version(), "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
           "transformers": transformers.__version__, "peft": peft.__version__, "cpu_count": os.cpu_count()}
    try:
        import psutil
        env["ram_total_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
    except Exception:
        pass
    try:
        du = shutil.disk_usage(root)
        env["drive_total_gb"], env["drive_free_gb"] = round(du.total / 2**30), round(du.free / 2**30)
        du = shutil.disk_usage("/content") if os.path.exists("/content") else du
        env["local_disk_free_gb"] = round(du.free / 2**30)
    except Exception:
        pass
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        env.update({"gpu": p.name, "gpu_mem_gb": round(p.total_memory / 2**30, 1),
                    "gpu_capability": f"{p.major}.{p.minor}", "gpu_sm_count": p.multi_processor_count})
        try:
            import pynvml
            pynvml.nvmlInit()
            d = pynvml.nvmlSystemGetDriverVersion()
            env["driver"] = d.decode() if isinstance(d, bytes) else d
            env["power_limit_w"] = pynvml.nvmlDeviceGetEnforcedPowerLimit(pynvml.nvmlDeviceGetHandleByIndex(0)) / 1000
        except Exception:
            pass
    env.update(extra or {})
    log_event(event="session_start", env=env)
    if _LOG["dir"] is not None:
        atomic_json(env, _LOG["dir"] / f"env_snapshot_{_LOG['session']}.json")
    return env


class ResourceMonitor:
    """Sampling di thread terpisah: utilisasi/memori/daya/suhu GPU (NVML) + RAM proses & sistem (psutil).
    Energi (Wh) = integral daya terhadap waktu. Semua gagal-aman: bila NVML/psutil tidak ada, kolom kosong."""

    def __init__(self, interval=5.0, index=0):
        self.interval, self.index = interval, index
        self._nv = self._h = self._ps = None
        self._lock = threading.Lock()
        self._stop = threading.Event()
        self._t = None

    def start(self):
        try:
            import pynvml
            pynvml.nvmlInit()
            self._nv, self._h = pynvml, pynvml.nvmlDeviceGetHandleByIndex(self.index)
        except Exception:
            self._nv = None
        try:
            import psutil
            self._ps = psutil
            self._proc = psutil.Process()
        except Exception:
            self._ps = None
        self.t0 = self._last = time.time()
        self.n = 0
        self.s = {"util": 0.0, "power": 0.0}      # akumulasi berbobot waktu
        self.mx = {"util": 0.0, "power": 0.0, "temp": 0.0, "gmem": 0.0, "pram": 0.0, "sram": 0.0}
        self.energy_ws = 0.0
        self._stop.clear()
        self._t = threading.Thread(target=self._loop, daemon=True)
        self._t.start()
        return self

    def _loop(self):
        while True:
            self._sample()
            if self._stop.wait(self.interval):
                break

    def _sample(self):
        now = time.time()
        with self._lock:
            dt = now - self._last
            self._last = now
            self.n += 1
            if self._nv is not None:
                try:
                    u = self._nv.nvmlDeviceGetUtilizationRates(self._h).gpu
                    pw = self._nv.nvmlDeviceGetPowerUsage(self._h) / 1000.0
                    tp = self._nv.nvmlDeviceGetTemperature(self._h, self._nv.NVML_TEMPERATURE_GPU)
                    gm = self._nv.nvmlDeviceGetMemoryInfo(self._h).used / 2**20
                    self.s["util"] += u * dt
                    self.s["power"] += pw * dt
                    self.energy_ws += pw * dt
                    self.mx["util"], self.mx["power"] = max(self.mx["util"], u), max(self.mx["power"], pw)
                    self.mx["temp"], self.mx["gmem"] = max(self.mx["temp"], tp), max(self.mx["gmem"], gm)
                except Exception:
                    pass
            if self._ps is not None:
                try:
                    self.mx["pram"] = max(self.mx["pram"], self._proc.memory_info().rss / 2**20)
                    self.mx["sram"] = max(self.mx["sram"], self._ps.virtual_memory().used / 2**20)
                except Exception:
                    pass

    def snapshot(self):
        with self._lock:
            wall = max(time.time() - self.t0, 1e-9)
            has = self._nv is not None
            return {"wall_seconds": wall, "n_samples": self.n, "n_sessions": 1,
                    "gpu_util_mean": self.s["util"] / wall if has else None, "gpu_util_max": self.mx["util"] if has else None,
                    "power_mean_w": self.s["power"] / wall if has else None, "power_max_w": self.mx["power"] if has else None,
                    "energy_wh": self.energy_ws / 3600.0 if has else None, "temp_max_c": self.mx["temp"] if has else None,
                    "gpu_mem_used_peak_mb": self.mx["gmem"] if has else None,
                    "proc_ram_peak_mb": self.mx["pram"] if self._ps else None,
                    "sys_ram_used_peak_mb": self.mx["sram"] if self._ps else None}

    def stop(self):
        if self._t is not None and self._t.is_alive():
            self._stop.set()
            self._t.join(timeout=10)
        return self.snapshot()


def merge_res(a, b):
    """Gabungkan ringkasan sumber daya antar-sesi (resume): jumlah untuk waktu/energi, maksimum untuk puncak,
    rata-rata berbobot waktu untuk utilisasi/daya."""
    if not a:
        return b
    out = {"wall_seconds": a["wall_seconds"] + b["wall_seconds"], "n_samples": a["n_samples"] + b["n_samples"],
           "n_sessions": a["n_sessions"] + b["n_sessions"]}
    for k in ("gpu_util_mean", "power_mean_w"):
        va, vb = a.get(k), b.get(k)
        out[k] = None if va is None and vb is None else ((va or 0) * a["wall_seconds"] + (vb or 0) * b["wall_seconds"]) / out["wall_seconds"]
    out["energy_wh"] = None if a.get("energy_wh") is None and b.get("energy_wh") is None else (a.get("energy_wh") or 0) + (b.get("energy_wh") or 0)
    for k in ("gpu_util_max", "power_max_w", "temp_max_c", "gpu_mem_used_peak_mb", "proc_ram_peak_mb", "sys_ram_used_peak_mb"):
        vs = [v for v in (a.get(k), b.get(k)) if v is not None]
        out[k] = max(vs) if vs else None
    return out


# ----------------------------------------------------------------------------- training
DIVERGE_TRANSIENT = True   # v4.6: (c) collapse sementara — Spearman validasi < ambang pada EPOCH MANA PUN (mis. 0,425 di epoch 2 lalu pulih)


def _diverged_from_hist(history, transient=None):
    """Collapse bila (a) Spearman validasi terbaik < DIVERGE_SPEARMAN, atau (b) jatuh >0,10 dari puncak sampai epoch terakhir
    (training runtuh setelah sempat membaik; mis. puncak 0,66 di epoch 1 lalu ~0,05)."""
    v = [h["val_spearman"] for h in history]
    tr = DIVERGE_TRANSIENT if transient is None else transient
    return bool(max(v) < DIVERGE_SPEARMAN or (max(v) - v[-1]) > 0.10 or (tr and min(v) < DIVERGE_SPEARMAN))


def is_diverged(result):
    return _diverged_from_hist(result["history"]) if result.get("history") else bool(result["val_spearman"] < DIVERGE_SPEARMAN)


def run_train(cfg: Config, root, train_df, val_df, device=None, log=print, ckpt_every=1, monitor_interval=5.0, scratch_root=None):
    """Latih satu konfigurasi dengan checkpoint per epoch. Aman dipanggil ulang (resume/skip).
    Mencatat waktu, memori, utilisasi, daya & energi ke result.json dan logs/compute_log.jsonl.
    scratch_root (disk lokal VM, mis. /content/sts_scratch): state optimizer per epoch dan bobot-terbaik sementara ditulis di sini,
    HANYA hasil akhir (best.pt, val_pred.npy, result.json) ditulis ke Drive -> bandwidth/operasi Drive turun >10x untuk Full FT
    (state AdamW XLM-R = 3,3 GB per epoch). Konsekuensi: run yang terputus di tengah diulang dari awal (maks. 1 run per putus sesi)."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_dir = Path(root) / cfg.run_id()
    run_dir.mkdir(parents=True, exist_ok=True)
    result_path = run_dir / "result.json"
    work = (Path(scratch_root) / cfg.run_id()) if scratch_root else run_dir
    work.mkdir(parents=True, exist_ok=True)
    state_path, best_path, valpred_path = work / "state.pt", work / "best.pt", work / "val_pred.npy"
    legacy_state = run_dir / "state.pt"          # state lama di Drive (run yang terputus sebelum memakai scratch)

    if result_path.exists():
        log(f"[SKIP] {cfg.run_id()} sudah selesai")
        log_event(event="skip", run_id=cfg.run_id())
        return json.loads(result_path.read_text())

    precision = pick_precision(device)
    set_seed(cfg.seed)
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    model = build_model(cfg).to(device)
    n_tr, n_tot = count_params(model)
    opt = build_optimizer(model, cfg)
    steps_per_epoch = (len(train_df) + cfg.batch_size - 1) // cfg.batch_size
    total = steps_per_epoch * cfg.epochs
    sched = get_linear_schedule_with_warmup(opt, int(cfg.warmup_ratio * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=(precision == "fp16"))
    gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"

    start_epoch, history, train_secs, peak_mem, peak_res, best_val, res_acc = 0, [], 0.0, 0.0, 0.0, -1.0, None
    load_state = state_path if state_path.exists() else (legacy_state if (scratch_root and legacy_state.exists()) else None)
    st = None
    if load_state is not None:
        try:
            st = _load_retry(load_state, "cpu", weights_only=False)
        except (EOFError, RuntimeError, pickle.UnpicklingError, ValueError) as e:
            # state.pt terpotong/rusak (mis. tulis terputus saat kuota Drive habis) -> buang, ulang run dari awal
            print(f"  [RUSAK] checkpoint {load_state.name} tidak terbaca ({e!r}); dihapus, run diulang dari epoch 0")
            for p_ in {load_state, state_path, legacy_state}:
                try: Path(p_).unlink()
                except OSError: pass
            st = None
    if st is not None:
        model.load_state_dict(st["weights"], strict=False)
        opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        scaler.load_state_dict(st["scaler"])
        start_epoch, history = st["epoch"], st["history"]
        train_secs, peak_mem, best_val = st["train_secs"], st["peak_mem"], st["best_val"]
        peak_res, res_acc = st.get("peak_res", 0.0), st.get("res_acc")
        random.setstate(st["rng_py"])
        np.random.set_state(st["rng_np"])
        torch.set_rng_state(st["rng_torch"])
        if device.type == "cuda":
            torch.cuda.set_rng_state_all(st["rng_cuda"])
        log(f"[RESUME] {cfg.run_id()} dari epoch {start_epoch}/{cfg.epochs}")
        log_event(event="resume", run_id=cfg.run_id(), epoch=start_epoch, gpu=gpu_name)
    else:
        log(f"[START] {cfg.run_id()} | trainable={n_tr:,}/{n_tot:,} | {precision}")
        log_event(event="start", run_id=cfg.run_id(), trainable=n_tr, total=n_tot, precision=precision, gpu=gpu_name)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    mon = ResourceMonitor(monitor_interval).start()

    try:
        for epoch in range(start_epoch, cfg.epochs):
            model.train()
            t0 = time.time()
            g = np.random.default_rng(cfg.seed * 1000 + epoch)      # urutan data deterministik per epoch
            order = g.permutation(len(train_df))
            tl = 0.0
            for s1, s2, y in _batches(train_df, cfg.batch_size, order):
                e1 = tok(s1, padding=True, truncation=True, max_length=cfg.max_len, return_tensors="pt").to(device)
                e2 = tok(s2, padding=True, truncation=True, max_length=cfg.max_len, return_tensors="pt").to(device)
                y = y.to(device)
                with _autocast(precision, device):
                    u, v = model.embed(e1), model.embed(e2)
                u, v = u.float(), v.float()
                loss = cosent_loss(u, v, y, cfg.cosent_scale) if cfg.strategy == "S2" else cosine_mse_loss(u, v, y)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                tl += loss.item()
            if device.type == "cuda":
                torch.cuda.synchronize()
            ep_secs = time.time() - t0
            train_secs += ep_secs
            if device.type == "cuda":
                peak_mem = max(peak_mem, torch.cuda.max_memory_allocated() / 2**20)
                peak_res = max(peak_res, torch.cuda.max_memory_reserved() / 2**20)

            pv = predict(model, tok, val_df, cfg, device, precision)
            m = metrics(pv, val_df["label"].values)
            history.append({"epoch": epoch + 1, "loss": tl / steps_per_epoch, "epoch_seconds": ep_secs,
                            "samples_per_s": len(train_df) / ep_secs, **{f"val_{k}": v for k, v in m.items()}})
            log(f"  epoch {epoch + 1}/{cfg.epochs} loss={tl / steps_per_epoch:.4f} val_spearman={m['spearman']:.4f} ({ep_secs:.0f}s)")
            log_event(event="epoch", run_id=cfg.run_id(), epoch=epoch + 1, epoch_seconds=round(ep_secs, 2),
                      val_spearman=m["spearman"], loss=tl / steps_per_epoch, peak_alloc_mb=peak_mem)
            if m["spearman"] > best_val:
                best_val = m["spearman"]
                atomic_save(trainable_state(model), best_path)
                np.save(valpred_path, pv)
            if (epoch + 1) % ckpt_every == 0 or epoch + 1 == cfg.epochs:
                atomic_save({"weights": trainable_state(model), "opt": opt.state_dict(), "sched": sched.state_dict(),
                             "scaler": scaler.state_dict(), "epoch": epoch + 1, "history": history,
                             "train_secs": train_secs, "peak_mem": peak_mem, "peak_res": peak_res, "best_val": best_val,
                             "res_acc": merge_res(res_acc, mon.snapshot()),
                             "rng_py": random.getstate(), "rng_np": np.random.get_state(),
                             "rng_torch": torch.get_rng_state(),
                             "rng_cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else None},
                            state_path)
    finally:
        seg = mon.stop()

    resources = merge_res(res_acc, seg)
    best_h = max(history, key=lambda h: h["val_spearman"])
    result = {"run_id": cfg.run_id(), "stage": _LOG["stage"], "config": asdict(cfg), "best_epoch": int(best_h["epoch"]),
              "val_spearman": best_val, "val_pearson": best_h["val_pearson"],
              "diverged": _diverged_from_hist(history),
              "history": history, "trainable_params": n_tr, "total_params": n_tot,
              "peak_gpu_mem_mb": peak_mem, "peak_gpu_mem_reserved_mb": peak_res,
              "train_seconds": train_secs, "wall_seconds": resources["wall_seconds"],
              "samples_per_s": len(train_df) * cfg.epochs / max(train_secs, 1e-9),
              "checkpoint_mb": best_path.stat().st_size / 2**20, "precision": precision,
              "resources": resources, "env": env_info(device)}
    if scratch_root:                              # satu kali tulis ke Drive: bobot terbaik + prediksi validasi, lalu hasil
        copy_to_drive(best_path, run_dir / "best.pt")
        if valpred_path.exists():
            copy_to_drive(valpred_path, run_dir / "val_pred.npy")
    atomic_json(result, result_path)
    log_event(event="finish", run_id=cfg.run_id(), val_spearman=best_val, train_seconds=round(train_secs, 1),
              wall_seconds=round(resources["wall_seconds"], 1), energy_wh=resources.get("energy_wh"), gpu=gpu_name)
    state_path.unlink(missing_ok=True)       # hemat ruang: state optimizer tidak dibutuhkan lagi
    if scratch_root:
        legacy_state.unlink(missing_ok=True)
        shutil.rmtree(work, ignore_errors=True)
    return result


# ----------------------------------------------------------------------------- evaluasi
def eval_run(cfg: Config, root, df, tag, device=None, log=print):
    """Evaluasi bobot terbaik sebuah run pada dataset `df` (mis. test EN, transfer ID). Hasil di-cache."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_dir = Path(root) / cfg.run_id()
    out_json, out_npy = run_dir / f"eval_{tag}.json", run_dir / f"pred_{tag}.npy"
    if out_json.exists():
        return json.loads(out_json.read_text())
    precision = pick_precision(device)
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    model = build_model(cfg).to(device)
    model.load_state_dict(_load_retry(run_dir / "best.pt", device), strict=False)
    pred = predict(model, tok, df, cfg, device, precision)
    res = {"tag": tag, "n": len(df), **metrics(pred, df["label"].values)}
    np.save(out_npy, pred)
    atomic_json(res, out_json)
    log(f"[EVAL] {cfg.run_id()} {tag}: spearman={res['spearman']:.4f}")
    return res


def eval_multi(cfg: Config, root, dfs: dict, device=None, log=print):
    """Evaluasi bobot terbaik sebuah run pada BEBERAPA dataset sekaligus (model dimuat sekali). Hasil di-cache."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_dir = Path(root) / cfg.run_id()
    todo = [t for t in dfs if not (run_dir / f"eval_{t}.json").exists()]
    if not todo:
        return
    precision = pick_precision(device)
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    model = build_model(cfg).to(device)
    model.load_state_dict(_load_retry(run_dir / "best.pt", device), strict=False)
    for tag in todo:
        pred = predict(model, tok, dfs[tag], cfg, device, precision)
        res = {"tag": tag, "n": len(dfs[tag]), **metrics(pred, dfs[tag]["label"].values)}
        np.save(run_dir / f"pred_{tag}.npy", pred)
        atomic_json(res, run_dir / f"eval_{tag}.json")
    log_event(event="eval", run_id=cfg.run_id(), tags=todo)
    del model


# ----------------------------------------------------------------------------- statistik
def paired_bootstrap_diff(pred_a, pred_b, y, n_boot=10000, seed=0):
    """Selisih Spearman (a - b) pada contoh uji yang sama; kembalikan (selisih, CI95)."""
    rng = np.random.default_rng(seed)
    n = len(y)
    d = np.empty(n_boot)
    for i in range(n_boot):
        ix = rng.integers(0, n, n)
        d[i] = spearmanr(pred_a[ix], y[ix])[0] - spearmanr(pred_b[ix], y[ix])[0]
    return float(spearmanr(pred_a, y)[0] - spearmanr(pred_b, y)[0]), tuple(np.percentile(d, [2.5, 97.5]))


def tost_paired(diffs, delta):
    """TOST berpasangan (Lakens, 2017) pada selisih per-seed; kembalikan p_TOST (maks dua uji satu sisi)."""
    from scipy.stats import ttest_1samp

    d = np.asarray(diffs, float)
    p_lo = ttest_1samp(d, -delta, alternative="greater").pvalue
    p_hi = ttest_1samp(d, delta, alternative="less").pvalue
    return float(max(p_lo, p_hi))


def holm(pvals):
    """Koreksi Holm-Bonferroni (Holm, 1979); kembalikan p yang disesuaikan sesuai urutan input."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj.tolist()


def _spearman_rows(P, y):
    """Spearman tiap baris P (m x n) terhadap y (n) -- vektorisasi via peringkat."""
    rp = rankdata(P, axis=1)
    ry = rankdata(y)
    rp = rp - rp.mean(1, keepdims=True)
    ry = ry - ry.mean()
    return (rp @ ry) / (np.sqrt((rp ** 2).sum(1)) * np.sqrt((ry ** 2).sum()) + 1e-12)


def boot_mean_diff(PA, PB, y, n_boot=5000, seed=0):
    """Bootstrap atas PASANGAN KALIMAT uji: selisih rerata Spearman (A - B), dirata-ratakan lintas seed.
    PA (kA x n), PB (kB x n) = prediksi tiap seed. Kembalikan (selisih titik, (CI95 bawah, atas))."""
    PA, PB, y = np.atleast_2d(PA), np.atleast_2d(PB), np.asarray(y)
    n = len(y)
    rng = np.random.default_rng(seed)
    point = float(_spearman_rows(PA, y).mean() - _spearman_rows(PB, y).mean())
    d = np.empty(n_boot)
    for b in range(n_boot):
        ix = rng.integers(0, n, n)
        d[b] = _spearman_rows(PA[:, ix], y[ix]).mean() - _spearman_rows(PB[:, ix], y[ix]).mean()
    return point, (float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))


def boot_drop_rel(P_en, y_en, P_id, y_id, n_boot=2000, seed=0):
    """Penurunan relatif transfer Drel = (rho_EN - rho_ID)/rho_EN pada PASANGAN PARALEL (indeks sama di EN & ID).
    Kembalikan (rho_EN, rho_ID, Drel, (CI95 bawah, atas))."""
    P_en, P_id, y_en, y_id = np.atleast_2d(P_en), np.atleast_2d(P_id), np.asarray(y_en), np.asarray(y_id)
    n = len(y_en)
    rng = np.random.default_rng(seed)

    def stat(ix):
        e, i = _spearman_rows(P_en[:, ix], y_en[ix]).mean(), _spearman_rows(P_id[:, ix], y_id[ix]).mean()
        return e, i, (e - i) / e

    e0, i0, d0 = stat(np.arange(n))
    ds = np.array([stat(rng.integers(0, n, n))[2] for _ in range(n_boot)])
    return float(e0), float(i0), float(d0), (float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5)))


def tost_table(d, delta):
    """Uji ekuivalensi TOST berpasangan (Lakens, 2017) pada selisih per-seed d. Mengembalikan tabel perhitungan lengkap."""
    from scipy.stats import t as tdist
    d = np.asarray(d, float)
    n = len(d)
    m, sd = d.mean(), d.std(ddof=1)
    se = sd / np.sqrt(n)
    t1, t2 = (m + delta) / se, (m - delta) / se
    p1, p2 = 1 - tdist.cdf(t1, n - 1), tdist.cdf(t2, n - 1)
    tc = tdist.ppf(0.975, n - 1)
    return {"n": n, "mean_diff": m, "sd_diff": sd, "se": se, "ci95_lo": m - tc * se, "ci95_hi": m + tc * se,
            "t_lower": t1, "t_upper": t2, "p_lower": p1, "p_upper": p2, "p_tost": max(p1, p2)}


def collect_results(root) -> pd.DataFrame:
    rows = []
    for f in Path(root).glob("*/result.json"):
        r = json.loads(f.read_text())
        c = r["config"]
        rs = r.get("resources") or {}
        row = {"run_id": r["run_id"], "stage": r.get("stage"), "model": c["model_name"], "method": c["method"],
               "rank": c["rank"], "strategy": c["strategy"], "lr": c["lr"], "seed": c["seed"], "epochs": c["epochs"],
               "rslora": c.get("rslora", False), "lora_targets": c.get("lora_targets", "all-linear"),
               "val_spearman": r["val_spearman"], "diverged": is_diverged(r),
               "diverged_orig": (_diverged_from_hist(r["history"], transient=False) if r.get("history") else bool(r["val_spearman"] < DIVERGE_SPEARMAN)),
               "val_min": (min(h["val_spearman"] for h in r["history"]) if r.get("history") else None),
               "best_epoch": r.get("best_epoch"), "trainable_params": r["trainable_params"],
               "total_params": r.get("total_params"), "peak_gpu_mem_mb": r["peak_gpu_mem_mb"],
               "peak_gpu_mem_reserved_mb": r.get("peak_gpu_mem_reserved_mb"),
               "train_seconds": r["train_seconds"], "wall_seconds": r.get("wall_seconds", r["train_seconds"]),
               "epoch_seconds": r["train_seconds"] / c["epochs"], "samples_per_s": r.get("samples_per_s"),
               "checkpoint_mb": r["checkpoint_mb"], "precision": r["precision"], "gpu": r["env"]["gpu"],
               "gpu_util_mean": rs.get("gpu_util_mean"), "power_mean_w": rs.get("power_mean_w"),
               "energy_wh": rs.get("energy_wh"), "proc_ram_peak_mb": rs.get("proc_ram_peak_mb"),
               "n_sessions": rs.get("n_sessions")}
        for e in f.parent.glob("eval_*.json"):
            ev = json.loads(e.read_text())
            row[f"{ev['tag']}_spearman"] = ev["spearman"]
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
sys.path.insert(0, f'{ROOT}/code')
import importlib, sts_lora_lib as L
importlib.reload(L)
import json, time, numpy as np, pandas as pd
from pathlib import Path
RUNS = f'{ROOT}/runs'; STAGES = f'{ROOT}/stages'; TABLES = f'{ROOT}/tables_v4b'; FIGS = f'{ROOT}/figures_v4b'
import shutil as _sh
os.makedirs(TABLES, exist_ok=True); os.makedirs(FIGS, exist_ok=True)
if os.path.isdir(f'{ROOT}/tables_v4'):   # v4.6: bawa tabel tahap awal (T03/T04/T04b ...) dari v4.5; tabel analisis akan ditulis ulang
    _sh.copytree(f'{ROOT}/tables_v4', TABLES, dirs_exist_ok=True)
DEVICE = torch.device('cuda')
L.set_log_dir(ROOT)                                  # log peristiwa -> {ROOT}/logs/compute_log.jsonl
ENV = L.log_session_start(ROOT)                      # snapshot GPU/driver/RAM/disk/versi -> logs/env_snapshot_<sesi>.json
print(json.dumps(ENV, indent=1))
def stage_done(name): return os.path.exists(f'{STAGES}/{name}.done')
def mark_done(name, info=''): open(f'{STAGES}/{name}.done', 'w').write(str(info))
try:
    from google.colab import userdata                # token HF opsional: Colab > ikon kunci > secret 'HF_TOKEN'
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
SCRATCH = '/content/sts_scratch'                       # disk LOKAL VM: state optimizer & bobot sementara (bukan Drive)
os.makedirs(SCRATCH, exist_ok=True)
def need_main():
    assert MAIN_OK, 'GPU tidak sesuai EXPECTED_GPU: tahap utama dihentikan agar data tidak tercampur antar-GPU'
def run_many(cfgs, train, val):
    need_main(); out = []
    for i, c in enumerate(cfgs, 1):
        print(f'--- {i}/{len(cfgs)}')
        try: out.append(L.run_train(c, RUNS, train, val, DEVICE, scratch_root=SCRATCH))
        except OSError as e:
            print('[DRIVE] kuota/I/O Drive bermasalah:', repr(e), '-> progres run selesai aman; tunggu (kuota Drive reset ±24 jam) lalu jalankan ulang sel ini'); raise
        except torch.cuda.OutOfMemoryError:
            print('[OOM]', c.run_id(), '-> kurangi batch_size & catat sebagai temuan'); torch.cuda.empty_cache()
    return out

## Pengaturan eksperimen (tetapkan sekali)
**Keterangan** — *Library:* — | *Algoritma:* desain faktorial 2 model × 5 metode (Full, LoRA r ∈ {4, 8, 16, 32}) × 3 strategi (S0, S1, S2) | *Model:* `xlm-roberta-base` (utama) dan `roberta-base` (pembanding Inggris), keduanya hanya MLM (tanpa kebocoran NLI/STS) | *Alasan:* grid LR LoRA digeser ke rentang yang lebih aman (1e-4…1e-3) dan dicari per rank karena dengan skala α/r konstan (=2) batas LR stabil menyempit saat rank membesar; `DELTA = 0,01` ditetapkan **sebelum** konfirmasi (margin ekuivalensi praktis: selisih Spearman 0,01 dianggap tidak bermakna untuk STS-B); seed konfirmasi 1–5 dan tambahan 6–10 terpisah dari seed pencarian/screening (42).

In [ ]:
MODELS   = ['xlm-roberta-base', 'roberta-base']
RANKS    = [4, 8, 16, 32]
STRATS   = ['S0', 'S1', 'S2']
EPOCHS   = 4
BATCH    = 32
SEED_SEARCH_SCREEN = 42                 # pencarian LR & screening
SEEDS_CONFIRM = [1, 2, 3, 4, 5]         # konfirmasi: SEMUA konfigurasi (30 x 5 = 150 run). Opsi 10 seed untuk SEMUA: [1..10] dan SEEDS_EXTRA = []
SEEDS_EXTRA   = [6, 7, 8, 9, 10]        # tambahan: kontras utama (Full terbaik vs LoRA terbaik) -> 10 seed
SEEDS_ABL     = [1, 2, 3]               # ablasi
ABL_RS_RANKS, ABL_QV_RANKS = (16, 32), (8, 16)   # rank untuk rsLoRA dan LoRA q,v-saja
LR_FULL  = [1e-5, 2e-5, 3e-5, 5e-5]              # grid Full FT (dicari per model, S0); tidak diperluas lagi: Full FT mendatar (plateau) di 3e-5..5e-5 dan mulai tidak stabil di atasnya
LR_LORA  = [5e-5, 1e-4, 2e-4, 3e-4, 5e-4, 1e-3]  # grid LoRA (dicari PER RANK, per model, S0)
LR_LORA_HI = {4: [2e-3], 8: [2e-3]}               # rank kecil terpilih di tepi atas (1e-3) -> grid diperluas hanya untuk r = 4, 8 (r >= 16 collapse di >= 5e-4)
LR_LORA_RANK = {rk: sorted(set(LR_LORA + LR_LORA_HI.get(rk, []))) for rk in RANKS}
PROBE_SEEDS = [43, 44]                  # uji stabilitas LR (Tahap 4b): seed ini TIDAK dipakai untuk hasil
DELTA    = 0.01                         # margin ekuivalensi TOST (ditetapkan sebelum konfirmasi)
ALPHA    = 0.05                         # tingkat signifikansi
NBOOT_MATCHED, NBOOT_BEST = 2000, 10000 # jumlah resampel bootstrap
def is_div(x): return L.is_diverged(x)   # collapse = Spearman validasi terbaik < 0,50 ATAU jatuh > 0,10 dari puncak sampai epoch terakhir
MAX_DIV_RATE_FOR_BEST = 0.20            # konfigurasi dengan laju divergensi > 20 % tidak boleh jadi "terbaik"
EXPECTED_GPU = 'L4'                     # kunci jenis GPU utama
UNIT_RATE = {'L4': 1.54, 'T4': None, 'A100': None, 'G4': None, 'H100': None}   # compute units/jam (hasil pengamatan panel Colab)

In [ ]:
gpu_name = torch.cuda.get_device_name(0)
MAIN_OK = (EXPECTED_GPU is None) or (EXPECTED_GPU in gpu_name)
print('GPU aktif:', gpu_name, '| presisi otomatis:', L.pick_precision(DEVICE), '| tahap utama diizinkan:', MAIN_OK)
if not MAIN_OK:
    print(f'PERINGATAN: GPU aktif bukan {EXPECTED_GPU}. Semua tahap utama diblokir agar data tidak tercampur antar-GPU.')
# Catat sisa compute units SECARA MANUAL (panel resources Colab) sebelum & sesudah tahap besar, mis.: L.log_units(87.5, 'sebelum pencarian LR')

## Diagnostik penyimpanan dan kuota Drive (baru di v4.4)
**Keterangan**
- *Library:* `shutil`, `subprocess` (`du` hanya pada folder **lokal** VM, tidak pernah pada Drive agar tidak menambah operasi Drive).
- *Algoritma:* membaca sisa disk VM (`Disk` di panel Resources, **bukan** kuota 1 TB Drive) dan ukuran folder cache lokal.
- *Model:* —
- *Alasan:* ada tiga batas yang berbeda: (1) **ruang Drive** (1 TB, aman); (2) **kuota operasi/bandwidth Drive** per pengguna dan per berkas, yang memicu pesan "Google Drive quota has been exceeded" dan galat `Input/output error`, tanpa angka batas resmi dan pulih sekitar sehari menurut FAQ Colab; (3) **disk VM** (±235 GB), yang tidak bertambah dengan membeli Drive. Pada v4.2 setiap epoch Full FT XLM-R menulis ±3,3 GB (bobot + state AdamW) ke Drive, sehingga total tulis puluhan hingga ratusan GB per tahap dan menembus kuota. v4.4 menulis state ke disk lokal dan hanya menyalin hasil akhir ke Drive.

In [ ]:
import shutil, subprocess
du = shutil.disk_usage('/content' if os.path.exists('/content') else '/')
print(f'Disk VM: terpakai {du.used / 2**30:.0f} / {du.total / 2**30:.0f} GB | bebas {du.free / 2**30:.0f} GB')
for p in ['/root/.config/Google', '/root/.cache/huggingface', SCRATCH, '/tmp']:
    if os.path.exists(p):
        print(f'{p}:', subprocess.run(['du', '-sh', p], capture_output=True, text=True).stdout.split()[0] if True else '')
if du.free < 40 * 2**30:
    print('PERINGATAN: disk VM hampir penuh. Pastikan unggahan Drive selesai (ikon Drive di panel kiri tanpa antrean), lalu Runtime > Disconnect and delete runtime, sambungkan lagi, dan Run all (run selesai dilewati).')

## Uji cepat (preflight) — sebelum pilot
**Keterangan**
- *Library:* `transformers`, `peft`, `torch`.
- *Algoritma:* membangun model SBERT penuh dan LoRA (r=8) lalu satu *forward pass* pada kalimat contoh.
- *Model:* `xlm-roberta-base`.
- *Alasan:* menangkap masalah lingkungan (konflik versi PEFT, GPU, presisi) dalam hitungan detik, sebelum menghabiskan unit. Juga memeriksa run lama dengan presisi berbeda.

In [ ]:
prec = L.pick_precision(DEVICE); print('GPU:', gpu_name, '| capability:', torch.cuda.get_device_capability(0), '| presisi:', prec)
tok = L.AutoTokenizer.from_pretrained('xlm-roberta-base')
for c in [L.Config(method='full'), L.Config(method='lora', rank=8)]:
    mod = L.build_model(c).to(DEVICE)
    tr, tot = L.count_params(mod)
    enc = tok(['kalimat uji pertama', 'a test sentence'], padding=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad(), L._autocast(prec, DEVICE): e = mod.embed(enc)
    print(c.method, f'trainable={tr:,}/{tot:,} ({100*tr/tot:.2f}%)', 'embedding', tuple(e.shape), 'finite:', bool(torch.isfinite(e).all()))
    del mod; torch.cuda.empty_cache()
stale = [json.load(open(f))['run_id'] for f in Path(RUNS).glob('*/result.json') if json.load(open(f))['precision'] != prec]
gp = {json.load(open(f))['env']['gpu'] for f in Path(RUNS).glob('*/result.json')}
print('Run dengan presisi berbeda:', stale if stale else 'tidak ada', '| GPU pada run yang sudah ada:', gp if gp else '-')

## Tahap 1 — Data
**Keterangan**
- *Library:* `datasets` (Hugging Face), `pandas`.
- *Algoritma:* normalisasi label ke [0,1]; verifikasi ukuran split dan **keselarasan pasangan** STS-B Inggris vs terjemahan Indonesia (selisih label kecil dan korelasi tinggi).
- *Model:* —
- *Alasan:* **STS-B** (Cer dkk., 2017; `sentence-transformers/stsb`) adalah tolok ukur STS standar; **LazarusNLP/stsb_mt_id** adalah terjemahan mesin STS-B (skor asli di kolom `correlation`, bukan `score`); **SemRel2024** (Ousidhoum dkk., 2024) mengukur *relatedness* (bukan similarity) dan dipakai sebagai evaluasi sekunder. Data Indonesia tidak punya split latih → latih di Inggris, uji di Indonesia. Selisih label kecil antara EN dan ID diperkirakan akibat pembulatan; toleransi 0,01 menolak ketidaksejajaran urutan. **Periksa lisensi dataset sebelum publikasi** (kartu dataset tidak menyebutkan lisensi).

In [ ]:
D = {}
D['train_en'] = L.load_data('stsb_en', 'train')
D['val_en']   = L.load_data('stsb_en', 'validation')
D['test_en']  = L.load_data('stsb_en', 'test')
D['test_id']  = L.load_data('stsb_id', 'test')
D['sr_en_test'] = L.load_data('semrel_en', 'test')
D['sr_id_test'] = L.load_data('semrel_id', 'test')
exp = {'train_en': 5749, 'val_en': 1500, 'test_en': 1379, 'test_id': 1379}
for k, n in exp.items(): assert len(D[k]) == n, (k, len(D[k]), n)
a, b = D['test_en']['label'].values, D['test_id']['label'].values
d = np.abs(a - b); r = np.corrcoef(a, b)[0, 1]
print(f'selisih label: maks={d.max():.4f} rata-rata={d.mean():.5f} | pasangan selisih>0.01: {(d > 0.01).sum()} | korelasi={r:.5f}')
i = int(d.argmax()); print('selisih terbesar ->', D['test_en'].iloc[i]['sentence1'][:70], '||', D['test_id'].iloc[i]['sentence1'][:70])
ALIGNED = bool(d.max() < 0.01 and r > 0.999)
print('PASANGAN PARALEL TERVERIFIKASI' if ALIGNED else 'PERINGATAN: pasangan EN/ID tidak selaras -> penurunan relatif hanya boleh dilaporkan agregat, bukan per pasangan')
for k, v in D.items(): print(k, len(v)); v.to_csv(f'{TABLES}/data_{k}.csv', index=False)
D['train_en'].head()

## Tahap 2 — Pilot dan estimasi biaya
**Keterangan**
- *Library:* `sts_lora_lib.run_train`.
- *Algoritma:* Full FT (S0, LR 2e-5) dan LoRA r=8 (S0, LR 3e-4), 1 seed, checkpoint per epoch, pemantauan sumber daya. Bila run identik dari v3 ada di Drive, dipakai ulang (langsung dilewati).
- *Model:* `xlm-roberta-base`.
- *Alasan:* mengukur waktu per epoch, memori, daya/energi **sebelum** menjalankan rangkaian penuh, lalu menghitung jumlah run dan perkiraan jam GPU/unit.

In [ ]:
need_main(); L.set_stage('pilot')
r = L.run_train(L.Config(model_name='xlm-roberta-base', method='full', strategy='S0', lr=2e-5, seed=SEED_SEARCH_SCREEN, epochs=EPOCHS, batch_size=BATCH), RUNS, D['train_en'], D['val_en'], DEVICE, scratch_root=SCRATCH)
r2 = L.run_train(L.Config(model_name='xlm-roberta-base', method='lora', rank=8, strategy='S0', lr=3e-4, seed=SEED_SEARCH_SCREEN, epochs=EPOCHS, batch_size=BATCH), RUNS, D['train_en'], D['val_en'], DEVICE, scratch_root=SCRATCH)
for name, x in [('full', r), ('lora r8', r2)]:
    rs = x['resources']
    print(name, {k: round(x[k], 3) for k in ['val_spearman', 'train_seconds', 'wall_seconds', 'peak_gpu_mem_mb', 'checkpoint_mb']},
          '| energi Wh', rs.get('energy_wh') and round(rs['energy_wh'], 1))
n_cfg = len(MODELS) * (1 + len(RANKS)) * len(STRATS)
plan = {'pencarian LR': len(MODELS) * (len(LR_FULL) + sum(len(LR_LORA_RANK[r_]) for r_ in RANKS)), 'screening (min.)': n_cfg,
        'uji stabilitas (min.)': n_cfg * len(PROBE_SEEDS), 'konfirmasi': n_cfg * len(SEEDS_CONFIRM), 'seed tambahan': len(MODELS) * 2 * len(SEEDS_EXTRA),
        'ablasi': (len(ABL_RS_RANKS) + len(ABL_QV_RANKS)) * len(SEEDS_ABL), 'benchmark': len(MODELS) * (1 + len(RANKS))}
avg = (r['wall_seconds'] + r2['wall_seconds']) / 2
tot = sum(plan.values()); hrs = tot * avg / 3600; rate = UNIT_RATE.get('L4')
print(plan, '| total run (maks.; run v3 identik dilewati):', tot)
print(f'perkiraan kasar: {hrs:.1f} jam GPU' + (f' ~ {hrs * rate:.0f} unit' if rate else ''))
mark_done('v4_2_pilot')

## Tahap 3 — Pencarian learning rate PER RANK (hanya data validasi)
**Keterangan**
- *Library:* `sts_lora_lib.run_train`, `pandas`.
- *Algoritma:* *grid search* pada S0, seed 42: Full FT (LR 1e-5, 2e-5, 3e-5) dan **LoRA untuk tiap rank** r ∈ {4, 8, 16, 32} (LR 1e-4, 3e-4, 5e-4, 1e-3), tiap model. LR terpilih = Spearman validasi tertinggi di antara run yang **tidak *collapse***. Seluruh tabel (LR × rank) disimpan sebagai bukti sensitivitas.
- *Model:* kedua model.
- *Alasan:* pada v3, LR hanya dicari di r=8 lalu dipakai di semua rank; run r=16/32 pada LR tersebut *collapse*. Dengan skala α/r = 2 konstan, LR stabil mengecil saat rank membesar (hipotesis; diuji lewat tabel ini dan ablasi rsLoRA). Pencarian per rank menghilangkan bias terhadap rank tertentu. Test tidak disentuh (Dodge dkk., 2019).

In [ ]:
need_main(); L.set_stage('lr_search')
lr_path = f'{STAGES}/lr_search_v4.json'
if not os.path.exists(lr_path):
    cfgs = []
    for m in MODELS:
        cfgs += [L.Config(model_name=m, method='full', strategy='S0', lr=lr, seed=SEED_SEARCH_SCREEN, epochs=EPOCHS, batch_size=BATCH) for lr in LR_FULL]
        for rk in RANKS:
            cfgs += [L.Config(model_name=m, method='lora', rank=rk, strategy='S0', lr=lr, seed=SEED_SEARCH_SCREEN, epochs=EPOCHS, batch_size=BATCH) for lr in LR_LORA_RANK[rk]]
    run_many(cfgs, D['train_en'], D['val_en'])
    dd_ = L.collect_results(RUNS); dd_['rank'] = dd_['rank'].astype(int)
    ids = {c.run_id() for c in cfgs}; g = dd_[dd_.run_id.isin(ids)]
    lr0 = {}
    for (m, me, rk), sub in g.groupby(['model', 'method', 'rank']):
        okk = sub[~sub.diverged]
        assert len(okk), f'semua LR collapse untuk {m} {me} r={rk}: perluas grid ke LR lebih kecil dan ulangi'
        lr0[f'{m}|{me}|{rk}'] = float(okk.loc[okk.val_spearman.idxmax(), 'lr'])
    json.dump(lr0, open(lr_path, 'w'), indent=2)
lr0 = json.load(open(lr_path))
dd_ = L.collect_results(RUNS); dd_['rank'] = dd_['rank'].astype(int)
gs = dd_[(dd_.seed == SEED_SEARCH_SCREEN) & (dd_.strategy == 'S0') & (~dd_.rslora) & (dd_.lora_targets == 'all-linear') & (dd_.epochs == EPOCHS)]
gs = gs[[(l_ in (LR_FULL if me == 'full' else LR_LORA_RANK[rk])) for l_, me, rk in zip(gs.lr, gs.method, gs['rank'])]]
T_LR = gs[['model', 'method', 'rank', 'lr', 'val_spearman', 'diverged']].sort_values(['model', 'method', 'rank', 'lr'])
T_LR['terpilih'] = [abs(lr0[f'{m}|{me}|{rk}'] - l_) < 1e-12 for m, me, rk, l_ in zip(T_LR.model, T_LR.method, T_LR['rank'], T_LR.lr)]
T_LR.to_csv(f'{TABLES}/T03_lr_search.csv', index=False)
print('LR awal terpilih:', lr0)
print('Run collapse pada pencarian LR:', int(T_LR.diverged.sum()), 'dari', len(T_LR), '(wajar: LR tinggi pada rank besar)')
edge = [k for k, v in lr0.items() if ('|lora|' in k and v == max(LR_LORA_RANK[int(k.split('|')[2])])) or (v == max(LR_FULL) and '|full|' in k)]
print('LR terpilih di tepi atas grid (catat sebagai keterbatasan):', edge or 'tidak ada')
piv = T_LR.pivot_table(index=['model', 'method', 'rank'], columns='lr', values='val_spearman'); piv.round(4)

In [ ]:
mark_done('v4_3_lr')

## Tahap 4 — Screening 30 konfigurasi + gerbang *collapse* otomatis
**Keterangan**
- *Library:* `sts_lora_lib.run_train`.
- *Algoritma:* tiap (model, metode, rank, strategi) dilatih 1 seed (42) pada LR hasil Tahap 3. Bila *collapse* (strategi S1/S2 bisa lebih sensitif daripada S0), LR **diturunkan satu langkah grid** dan dilatih ulang, berulang hingga stabil. Bila grid habis dan masih *collapse*, sel **berhenti dengan peringatan** (jeda otomatis). Hasil: `lr_final_v4.json` (LR final per konfigurasi) dan catatan *fallback*.
- *Model:* kedua model.
- *Alasan:* mencegah pengulangan masalah v3 (7 dari 40 run konfirmasi *collapse*) sebelum menghabiskan unit pada 150 run konfirmasi. Penurunan LR ditetapkan hanya dari **validasi**; seluruh keputusan tercatat sebagai bagian protokol.

In [ ]:
need_main(); L.set_stage('screening')
fb_path = f'{STAGES}/lr_final_v4.json'
if not os.path.exists(fb_path):
    lr_final, fallback = {}, []
    for m in MODELS:
        for s in STRATS:
            for me, rk in [('full', 0)] + [('lora', r_) for r_ in RANKS]:
                grid = sorted(LR_FULL if me == 'full' else LR_LORA_RANK[rk], reverse=True)
                lr = lr0[f'{m}|{me}|{rk}']; tried = []
                while True:
                    c = L.Config(model_name=m, method=me, rank=rk, strategy=s, lr=lr, seed=SEED_SEARCH_SCREEN, epochs=EPOCHS, batch_size=BATCH)
                    x = run_many([c], D['train_en'], D['val_en'])[0]
                    dv_ = is_div(x); tried.append((lr, round(x['val_spearman'], 4), dv_))
                    if not dv_: break
                    lower = [g_ for g_ in grid if g_ < lr - 1e-15]
                    if not lower:
                        raise RuntimeError(f'PAUSE: {m} {me} r={rk} {s} masih collapse di LR terkecil grid {tried}. Perluas grid ke LR lebih kecil, ulangi mulai Tahap 3.')
                    lr = lower[0]
                lr_final[f'{m}|{me}|{rk}|{s}'] = lr
                if len(tried) > 1: fallback.append({'model': m, 'method': me, 'rank': rk, 'strategy': s, 'lr_awal': lr0[f'{m}|{me}|{rk}'], 'lr_final': lr, 'percobaan': str(tried)})
    json.dump(lr_final, open(fb_path, 'w'), indent=2)
    pd.DataFrame(fallback).to_csv(f'{TABLES}/T03b_lr_fallback.csv', index=False)
lr_final = json.load(open(fb_path))
fbk = pd.read_csv(f'{TABLES}/T03b_lr_fallback.csv') if os.path.exists(f'{TABLES}/T03b_lr_fallback.csv') and os.path.getsize(f'{TABLES}/T03b_lr_fallback.csv') > 5 else pd.DataFrame()
if len(fbk): print('PERINGATAN: LR diturunkan (collapse terdeteksi pada validasi) untuk', len(fbk), 'konfigurasi:'); print(fbk.drop(columns='percobaan').to_string(index=False))
else: print('Tidak ada collapse pada screening: semua LR awal dipertahankan.')
def lr_for(m, me, rk, s): return lr_final[f'{m}|{me}|{rk}|{s}']
screen_cfgs = [L.Config(model_name=m, method=me, rank=rk, strategy=s, lr=lr_for(m, me, rk, s), seed=SEED_SEARCH_SCREEN, epochs=EPOCHS, batch_size=BATCH)
               for m in MODELS for s in STRATS for me, rk in [('full', 0)] + [('lora', r_) for r_ in RANKS]]
dd_ = L.collect_results(RUNS); dd_['rank'] = dd_['rank'].astype(int)
scr = dd_[dd_.run_id.isin({c.run_id() for c in screen_cfgs})].pivot_table(index=['model', 'method', 'rank'], columns='strategy', values='val_spearman')
scr.to_csv(f'{TABLES}/T04_screening_val_spearman.csv'); mark_done('v4_4_screening'); scr.round(4)

## Tahap 4b — Uji stabilitas LR pada 2 seed tambahan (baru di v4.2)
**Keterangan**
- *Library:* `sts_lora_lib.run_train`.
- *Algoritma:* tiap dari 30 konfigurasi dilatih pada LR final dengan seed **43 dan 44** (seed yang tidak dipakai untuk hasil). Bila salah satu *collapse* (lihat definisi di Rumus 9), LR diturunkan satu langkah grid dan seluruh konfigurasi itu (seed 42, 43, 44) diuji ulang; berulang hingga stabil pada ketiga seed. Bila grid habis → sel berhenti (PAUSE).
- *Model:* kedua model.
- *Alasan:* LR terpilih dari **satu seed** sering berada di tepi "jurang" *collapse* (mis. r=16 S2 collapse pada 5e-4 dan 3e-4; r=32 S2 sempat jatuh ke 0,68 di epoch 2). Seed baru hanya akan muncul di konfirmasi dan menyisakan lubang data. Uji 2 seed ekstra (≈ 60 run) jauh lebih murah daripada kehilangan 5 seed pada sebuah konfigurasi, dan membuat laju divergensi yang dilaporkan di naskah mencerminkan ketidakstabilan **sisa**, bukan salah pilih LR. Keputusan hanya memakai validasi.

In [ ]:
need_main(); L.set_stage('stability_probe')
rows_p = []
if not stage_done('v4_4b_probe'):
    for m in MODELS:
        for s in STRATS:
            for me, rk in [('full', 0)] + [('lora', r_) for r_ in RANKS]:
                grid = sorted(LR_FULL if me == 'full' else LR_LORA_RANK[rk], reverse=True); key = f'{m}|{me}|{rk}|{s}'; lr = lr_final[key]
                while True:
                    cs = [L.Config(model_name=m, method=me, rank=rk, strategy=s, lr=lr, seed=sd, epochs=EPOCHS, batch_size=BATCH) for sd in [SEED_SEARCH_SCREEN] + PROBE_SEEDS]
                    rs = run_many(cs, D['train_en'], D['val_en'])
                    dv_ = [is_div(x) for x in rs]
                    for x in rs: rows_p.append({'model': m, 'method': me, 'rank': rk, 'strategy': s, 'lr': lr, 'seed': x['config']['seed'], 'val': x['val_spearman'], 'collapse': is_div(x)})
                    if not any(dv_): break
                    lower = [g_ for g_ in grid if g_ < lr - 1e-15]
                    if not lower: raise RuntimeError(f'PAUSE: {key} collapse pada LR terkecil grid. Kirim pesan ini; grid perlu diperluas ke bawah.')
                    print(f'  >> {key}: collapse pada LR {lr:g} (seed {[x["config"]["seed"] for x, d_ in zip(rs, dv_) if d_]}) -> turun ke {lower[0]:g}'); lr = lower[0]
                lr_final[key] = lr
    json.dump(lr_final, open(fb_path, 'w'), indent=2)
    pd.DataFrame(rows_p).to_csv(f'{TABLES}/T04b_uji_stabilitas.csv', index=False); mark_done('v4_4b_probe')
lr_final = json.load(open(fb_path))
pr = pd.read_csv(f'{TABLES}/T04b_uji_stabilitas.csv'); TBL_PROBE = pr
print('Uji stabilitas selesai. Konfigurasi dengan LR berubah:', int((pr.groupby(['model', 'method', 'rank', 'strategy']).lr.nunique() > 1).sum()), '| run collapse selama uji:', int(pr.collapse.sum()), 'dari', len(pr))
dd_ = L.collect_results(RUNS); dd_['rank'] = dd_['rank'].astype(int)
screen_cfgs = [L.Config(model_name=m, method=me, rank=rk, strategy=s, lr=lr_for(m, me, rk, s), seed=SEED_SEARCH_SCREEN, epochs=EPOCHS, batch_size=BATCH)
               for m in MODELS for s in STRATS for me, rk in [('full', 0)] + [('lora', r_) for r_ in RANKS]]
scr = dd_[dd_.run_id.isin({c.run_id() for c in screen_cfgs})].pivot_table(index=['model', 'method', 'rank'], columns='strategy', values='val_spearman')
scr.to_csv(f'{TABLES}/T04_screening_val_spearman.csv'); print('LR final per konfigurasi sudah stabil pada seed 42, 43, 44.'); scr.round(4)

## Tahap 5 — Konfirmasi: SEMUA konfigurasi × 5 seed (150 run)
**Keterangan**
- *Library:* `sts_lora_lib.run_train`.
- *Algoritma:* 2 model × 5 metode × 3 strategi × seed {1,…,5}, LR final per konfigurasi. Run yang *collapse* **tidak diulang**: dicatat sebagai divergensi (hasil penelitian), dikeluarkan dari rerata.
- *Model:* kedua model.
- *Alasan:* perbandingan RQ1 pada strategi yang sama dan interaksi metode × strategi (RQ2) memerlukan data multi-seed untuk setiap sel desain; satu seed tidak cukup karena variansi antar-seed STS-B sebanding dengan selisih antar-metode. Seed konfirmasi terpisah dari seed screening.

In [ ]:
need_main(); L.set_stage('confirm')
conf_cfgs = [L.Config(model_name=m, method=me, rank=rk, strategy=s, lr=lr_for(m, me, rk, s), seed=sd, epochs=EPOCHS, batch_size=BATCH)
             for m in MODELS for s in STRATS for me, rk in [('full', 0)] + [('lora', r_) for r_ in RANKS] for sd in SEEDS_CONFIRM]
print(len(conf_cfgs), 'run konfirmasi')
run_many(conf_cfgs, D['train_en'], D['val_en'])
dd_ = L.collect_results(RUNS); dd_['rank'] = dd_['rank'].astype(int)
cf = dd_[dd_.run_id.isin({c.run_id() for c in conf_cfgs})]
dvt = cf.groupby(['model', 'method', 'rank', 'strategy']).agg(n=('seed', 'count'), n_collapse=('diverged', 'sum')).reset_index()
print('Run collapse pada konfirmasi:', int(cf.diverged.sum()), 'dari', len(cf))
print(dvt[dvt.n_collapse > 0].to_string(index=False) if cf.diverged.any() else 'Tidak ada collapse.')
mark_done('v4_5_confirm')

## Tahap 5b — Seed tambahan (10 seed) untuk kontras utama
**Keterangan**
- *Library:* `pandas`, `sts_lora_lib.run_train`.
- *Algoritma:* per model, **Full terbaik** dan **LoRA terbaik** dipilih dari rerata Spearman **validasi** pada seed 1–5 (hanya run tidak *collapse*; konfigurasi dengan laju divergensi > 20 % tidak boleh terpilih), lalu ditambah seed 6–10 sehingga masing-masing 10 seed.
- *Model:* kedua model.
- *Alasan:* dengan 5 seed, p dua sisi minimum uji Wilcoxon = 0,0625 dan daya TOST rendah; 10 seed membuat uji peringkat bertanda dapat mencapai p < 0,05 dan mempersempit CI selisih. Pemilihan memakai validasi saja (Dodge dkk., 2019).

In [ ]:
need_main(); L.set_stage('confirm_extra')
dd_ = L.collect_results(RUNS); dd_['rank'] = dd_['rank'].astype(int)
cf = dd_[dd_.run_id.isin({c.run_id() for c in conf_cfgs})]
grp = cf.groupby(['model', 'method', 'rank', 'strategy']).apply(lambda g: pd.Series({'n': len(g), 'div_rate': g.diverged.mean(), 'val': g.loc[~g.diverged, 'val_spearman'].mean()}), include_groups=False).reset_index()
grp = grp[(grp.div_rate <= MAX_DIV_RATE_FOR_BEST) & grp.val.notna()]
BEST = {}
for m in MODELS:
    gm = grp[grp.model == m]
    bf = gm[gm.method == 'full'].sort_values('val', ascending=False).iloc[0]
    bl = gm[gm.method == 'lora'].sort_values('val', ascending=False).iloc[0]
    BEST[m] = {'full': ['full', 0, bf.strategy], 'lora': ['lora', int(bl['rank']), bl.strategy]}
json.dump(BEST, open(f'{STAGES}/best_pairs_v4.json', 'w'), indent=2); print(json.dumps(BEST, indent=1))
extra_cfgs = [L.Config(model_name=m, method=BEST[m][k][0], rank=BEST[m][k][1], strategy=BEST[m][k][2], lr=lr_for(m, BEST[m][k][0], BEST[m][k][1], BEST[m][k][2]), seed=sd, epochs=EPOCHS, batch_size=BATCH)
              for m in MODELS for k in ('full', 'lora') for sd in SEEDS_EXTRA]
print(len(extra_cfgs), 'run seed tambahan'); run_many(extra_cfgs, D['train_en'], D['val_en']); mark_done('v4_5b_extra')

## Tahap 6 — Ablasi varian LoRA (wajib)
**Keterangan**
- *Library:* `peft` (`use_rslora`, `target_modules`).
- *Algoritma:* pada `xlm-roberta-base`, S0, seed 1–3: **rsLoRA** (skala γ = α/√r dengan α = 2√8, sehingga γ₈ = 2) pada r = 16 dan 32 (skala lebih kecil daripada LoRA standar); **LoRA q,v-saja** (konfigurasi LoRA asli) pada r = 8 dan 16. Dibandingkan dengan LoRA semua-lapisan-linear pada rank dan seed yang sama.
- *Model:* `xlm-roberta-base`.
- *Alasan:* menguji dua penjelasan: (i) apakah *collapse*/penurunan akurasi pada rank besar disebabkan skala α/r konstan (rsLoRA memperkecil skala pada r besar), dan (ii) apakah kesimpulan RQ1 bergantung pada pemilihan modul target. LR tidak dicari ulang (memakai LR S0 rank yang sama; batasan yang dilaporkan).

In [ ]:
need_main(); L.set_stage('ablation')
m_ab = 'xlm-roberta-base'
abl = []
for sd in SEEDS_ABL:
    for rk in ABL_RS_RANKS: abl.append(L.Config(model_name=m_ab, method='lora', rank=rk, strategy='S0', lr=lr_for(m_ab, 'lora', rk, 'S0'), seed=sd, epochs=EPOCHS, batch_size=BATCH, rslora=True))
    for rk in ABL_QV_RANKS: abl.append(L.Config(model_name=m_ab, method='lora', rank=rk, strategy='S0', lr=lr_for(m_ab, 'lora', rk, 'S0'), seed=sd, epochs=EPOCHS, batch_size=BATCH, lora_targets='qv'))
run_many(abl, D['train_en'], D['val_en']); mark_done('v4_6_ablation')

## Tahap 7 — Benchmark efisiensi GPU (wajib)
**Keterangan**
- *Library:* `sts_lora_lib.run_train`, pemantau sumber daya (NVML).
- *Algoritma:* 1 epoch untuk Full dan LoRA r ∈ {4, 8, 16, 32} (S0, seed 0), kedua model, pada GPU aktif; disimpan di `bench/<nama GPU>`.
- *Model:* kedua model.
- *Alasan:* ukuran waktu/memori/daya dalam kondisi terkontrol (1 epoch, seed tetap) sebagai pelengkap run konfirmasi; tabel ini menjadi dasar klaim efisiensi dan menyimpan jejak GPU yang dipakai.

In [ ]:
need_main(); L.set_stage('bench')
broot = f"{ROOT}/bench/{gpu_name.replace(' ', '_')}"; os.makedirs(broot, exist_ok=True)
bcfg = [L.Config(model_name=m, method=me, rank=rk, strategy='S0', lr=lr_for(m, me, rk, 'S0'), seed=0, epochs=1, batch_size=BATCH)
        for m in MODELS for me, rk in [('full', 0)] + [('lora', r_) for r_ in RANKS]]
for c in bcfg: L.run_train(c, broot, D['train_en'], D['val_en'], DEVICE, scratch_root=SCRATCH)
bt = L.collect_results(broot); bt['rank'] = bt['rank'].astype(int)
bt = bt.sort_values(['model', 'method', 'rank'])[['model', 'method', 'rank', 'epoch_seconds', 'samples_per_s', 'peak_gpu_mem_mb', 'peak_gpu_mem_reserved_mb', 'gpu_util_mean', 'power_mean_w', 'energy_wh', 'checkpoint_mb', 'trainable_params', 'precision', 'gpu']]
bt.to_csv(f'{TABLES}/T12_benchmark_gpu.csv', index=False); mark_done('v4_7_bench'); bt.round(2)

## Tahap 7b — Verifikasi kelengkapan run (baru di v4.3)
**Keterangan**
- *Library:* `pathlib`, `sts_lora_lib.run_train`.
- *Algoritma:* memeriksa bahwa setiap run konfirmasi, seed tambahan, dan ablasi punya `result.json` dan `best.pt` di Drive; run yang hilang dilatih ulang (run lengkap dilewati); bila `best.pt` hilang pada run yang sudah punya `result.json`, sel berhenti dengan daftar run itu.
- *Model:* —
- *Alasan:* pada v4.2, pembacaan Drive sempat gagal sesaat (FileNotFoundError) dan hanya 142 dari 150 run konfirmasi terbaca oleh `collect_results`. Pemeriksaan eksplisit memastikan data lengkap sebelum evaluasi dan statistik, sehingga tidak ada run yang "hilang" diam-diam dari analisis.

In [ ]:
need_main(); L.set_stage('verify')
all_cfgs = conf_cfgs + extra_cfgs + abl
rd = lambda c: Path(RUNS) / c.run_id()
miss = [c for c in all_cfgs if not (rd(c) / 'result.json').exists()]
nobest = [c for c in all_cfgs if (rd(c) / 'result.json').exists() and not (rd(c) / 'best.pt').exists()]
print(f'total run direncanakan: {len(all_cfgs)} | result.json hilang: {len(miss)} | best.pt hilang: {len(nobest)}')
if miss:
    print('melatih ulang run yang hilang:', [c.run_id() for c in miss]); run_many(miss, D['train_en'], D['val_en'])
if nobest:
    raise RuntimeError('PAUSE: best.pt hilang pada run berikut (hapus folder run itu di Drive lalu jalankan ulang sel ini): ' + ', '.join(c.run_id() for c in nobest))
dd_ = L.collect_results(RUNS); found = dd_.run_id.isin({c.run_id() for c in all_cfgs}).sum()
assert found == len(all_cfgs), f'collect_results hanya menemukan {found} dari {len(all_cfgs)} run: tunggu sinkronisasi Drive 1-2 menit, lalu jalankan ulang sel ini'
print('Semua run lengkap:', found); mark_done('v4_7b_verify')

## Tahap 8 — Evaluasi test Inggris dan transfer ke Indonesia (test dipakai sekali)
**Keterangan**
- *Library:* `sts_lora_lib.eval_multi`, `scipy.stats`.
- *Algoritma:* bobot terbaik (menurut validasi) tiap run diinferensi pada STS-B test (EN), SemRel2024 test (EN), STS-B test terjemahan (ID), SemRel2024 test (ID); Spearman & Pearson; prediksi disimpan (.npy) untuk *bootstrap* berpasangan. **Baseline tanpa fine-tuning** (mean pooling model asli) dihitung sebagai pembanding.
- *Model:* seluruh run konfirmasi, seed tambahan, dan ablasi.
- *Alasan:* test baru disentuh setelah semua pilihan (LR, strategi, konfigurasi terbaik) dikunci pada validasi. Transfer ke bahasa Indonesia dilakukan tanpa pelatihan tambahan (RQ3). Run *collapse* tidak dievaluasi: dikeluarkan dari statistik dan dilaporkan lewat laju divergensi (nilai validasinya tetap tercatat di `result.json`).

In [ ]:
need_main(); L.set_stage('eval')
EVALS = {'test_en': D['test_en'], 'sr_en_test': D['sr_en_test'], 'test_id': D['test_id'], 'sr_id_test': D['sr_id_test']}
dd_ = L.collect_results(RUNS); skip_ids = set(dd_[dd_.diverged].run_id)       # run collapse tidak dievaluasi (dikeluarkan dari statistik; hemat operasi Drive)
print('dilewati (collapse):', len(skip_ids))
for i, c in enumerate(conf_cfgs + extra_cfgs + abl, 1):
    if c.run_id() in skip_ids: continue
    L.eval_multi(c, RUNS, EVALS, DEVICE)
bpath = f'{TABLES}/baseline_no_finetune.json'
if not os.path.exists(bpath):
    base = {}
    for m in MODELS:
        c = L.Config(model_name=m, method='full'); tok = L.AutoTokenizer.from_pretrained(m)
        mod = L.build_model(c).to(DEVICE); prec = L.pick_precision(DEVICE)
        for tag, d_ in EVALS.items():
            p_ = L.predict(mod, tok, d_, c, DEVICE, prec); base[f'{m}|{tag}'] = L.metrics(p_, d_['label'].values)['spearman']; np.save(f'{TABLES}/basepred_{m}_{tag}.npy', p_)
        del mod; torch.cuda.empty_cache()
    json.dump(base, open(bpath, 'w'), indent=2)
base = json.load(open(bpath)); print({k: round(v, 4) for k, v in base.items()}); mark_done('v4_8_eval')

## Tahap 8b — Audit konsistensi bobot terbaik dan perbaikan otomatis
**Keterangan**
- *Library:* `sts_lora_lib.eval_run`, `sts_lora_lib.run_train`, `pandas`.
- *Algoritma:* untuk SETIAP run yang dipakai analisis (konfirmasi, seed tambahan, ablasi; bukan *collapse*), muat `best.pt`, hitung ulang Spearman validasi, bandingkan dengan nilai tersimpan dari latihan. |Δ| ≤ 0,002 = konsisten. Run tidak konsisten (bobot terbaik tidak sama dengan bobot pada epoch terbaik, mis. penyimpanan Drive gagal saat kuota habis) dipindahkan ke `runs_invalid/` (arsip, tidak dihapus), DILATIH ULANG dengan konfigurasi dan seed yang sama, dievaluasi ulang, lalu diaudit lagi.
- *Model matematika:* $\Delta_i = \rho_i^{\text{ulang}} - \rho_i^{\text{tersimpan}}$, konsisten bila $|\Delta_i| \le 0{,}002$.
- *Alasan:* pada v4.8 satu dari 21 run yang dicek (Full XLM-R S2 seed 3) tidak konsisten: validasi ulang 0,769 (sama dengan epoch 1) padahal tersimpan 0,859 (epoch 4). Jadi `best.pt`-nya usang dan angka test-nya tidak sah. Audit penuh memastikan tidak ada run lain yang terdampak. Hasil audit dan daftar run yang dilatih ulang menjadi lampiran naskah.

In [ ]:
need_main(); L.set_stage('audit')
ALL_CFGS = conf_cfgs + extra_cfgs + abl
def run_audit():
    d0 = L.collect_results(RUNS); ok_ids = set(d0[~d0.diverged].run_id); rows = []
    for c in ALL_CFGS:
        if c.run_id() not in ok_ids: continue
        ev = L.eval_run(c, RUNS, D['val_en'], 'val_audit', DEVICE, log=lambda *a, **k: None)
        st_ = float(d0[d0.run_id == c.run_id()].val_spearman.iloc[0])
        rows.append({'run_id': c.run_id(), 'model': c.model_name, 'method': c.method, 'rank': c.rank, 'strategy': c.strategy, 'lr': c.lr, 'seed': c.seed,
                     'val_tersimpan': st_, 'val_ulang': ev['spearman'], 'selisih': ev['spearman'] - st_})
    a_ = pd.DataFrame(rows); a_['konsisten'] = a_.selisih.abs() <= 0.002
    return a_
aud = run_audit()
print('Run diaudit:', len(aud), '| konsisten:', int(aud.konsisten.sum()), '| TIDAK konsisten:', int((~aud.konsisten).sum()))
print(aud[~aud.konsisten].round(4).to_string(index=False) if (~aud.konsisten).any() else '(semua konsisten)')

In [ ]:
need_main(); L.set_stage('repair')
bad_ids = list(aud[~aud.konsisten].run_id); repaired = []
INVALID = f'{ROOT}/runs_invalid'; os.makedirs(INVALID, exist_ok=True)
for c in ALL_CFGS:
    if c.run_id() not in bad_ids: continue
    src_ = f'{RUNS}/{c.run_id()}'; dst_ = f'{INVALID}/{c.run_id()}'
    if os.path.exists(dst_): shutil.rmtree(dst_)
    shutil.move(src_, dst_)                                    # arsip run lama (tidak dihapus)
    print('[REPAIR] dilatih ulang:', c.run_id())
    L.run_train(c, RUNS, D['train_en'], D['val_en'], DEVICE, scratch_root=SCRATCH)
    if not L.is_diverged(json.load(open(f'{RUNS}/{c.run_id()}/result.json'))): L.eval_multi(c, RUNS, EVALS, DEVICE)
    repaired.append(c.run_id())
if repaired:
    aud = run_audit();     print('Audit ulang -> TIDAK konsisten:', int((~aud.konsisten).sum()))
print('Run dilatih ulang:', len(repaired), repaired)

## Tahap 9 — Statistik RQ1–RQ3 (tabel perhitungan dan perbandingan)
**Keterangan**
- *Library:* `pandas`, `numpy`, `scipy.stats` (uji-t, Wilcoxon), fungsi `tost_table`, `boot_mean_diff`, `boot_drop_rel`, `holm` dari `sts_lora_lib`.
- *Algoritma:* hanya run **tidak *collapse*** yang dihitung; laju divergensi dilaporkan terpisah. **RQ1:** selisih berpasangan per seed Δ = ρ_LoRA − ρ_Full pada **strategi sama** (matched) untuk tiap model × strategi × rank, TOST (δ = 0,01), CI-t 95 %, *bootstrap* atas kalimat uji, koreksi Holm per famili; analisis kedua **terbaik vs terbaik** (10 seed, + Wilcoxon). **RQ2:** efek strategi terhadap S0 (uji-t berpasangan) dan interaksi metode × strategi (selisih dari selisih, Rumus 14). **RQ3:** ρ_ID, penurunan relatif Δrel (Rumus 15) dengan CI *bootstrap* pada pasangan paralel, serta perbandingan LoRA vs Full pada ρ_ID.
- *Model:* —
- *Alasan:* klaim utama adalah *ekuivalensi* (LoRA tidak lebih buruk dari δ), yang tidak dapat dibuktikan oleh uji beda biasa; CI dan p dilaporkan bersama ukuran efek. Koreksi Holm dikerjakan **per famili** agar tidak tercemar uji yang tidak setara. Dengan 5 seed, simpangan baku selisih ≈ 0,006 hanya memungkinkan ekuivalensi bila |Δ| ≲ 0,005; 10 seed ≲ 0,007 (karena itu kontras utama memakai 10 seed).

In [ ]:
need_main()
from scipy.stats import ttest_1samp, wilcoxon, t as tdist
df_all = L.collect_results(RUNS); df_all['rank'] = df_all['rank'].astype(int)
std = df_all[(~df_all.rslora) & (df_all.lora_targets == 'all-linear') & (df_all.epochs == EPOCHS)].copy()
std = std[[abs(l_ - lr_final.get(f'{m}|{me}|{rk}|{s}', -1)) < 1e-12 for l_, m, me, rk, s in zip(std.lr, std.model, std.method, std['rank'], std.strategy)]]
main = std[std.seed.isin(SEEDS_CONFIRM + SEEDS_EXTRA)].copy()
okm = main[~main.diverged].copy()                      # run valid (tidak collapse)
okc = okm[okm.seed.isin(SEEDS_CONFIRM)]                # 5 seed konfirmasi (analisis seragam)
TBL = {}                                               # nama tabel -> DataFrame (diekspor ke CSV/Excel)
EVTAG = {'test_en_spearman': 'test_en', 'test_id_spearman': 'test_id'}

def decide(row):
    if row['p_tost_holm'] < ALPHA: return 'Ekuivalen (|Δ| < δ)'
    if row['ci_hi'] < -DELTA: return 'LoRA lebih buruk'
    if row['ci_lo'] > DELTA: return 'LoRA lebih baik'
    return 'Tidak konklusif'

def pair_stats(m, f_strat, l_rank, l_strat, col, seeds, nboot, wilc=False):
    fu = okm[(okm.model == m) & (okm.method == 'full') & (okm.strategy == f_strat) & okm.seed.isin(seeds)].set_index('seed')
    lo = okm[(okm.model == m) & (okm.method == 'lora') & (okm['rank'] == l_rank) & (okm.strategy == l_strat) & okm.seed.isin(seeds)].set_index('seed')
    cm = sorted(set(fu.index) & set(lo.index))
    if len(cm) < 3: return None
    d = lo.loc[cm, col].values - fu.loc[cm, col].values
    tt = L.tost_table(d, DELTA)
    tag = EVTAG[col]; y = EVALS[tag]['label'].values
    PA = np.stack([np.load(f"{RUNS}/{lo.loc[s, 'run_id']}/pred_{tag}.npy") for s in cm])
    PB = np.stack([np.load(f"{RUNS}/{fu.loc[s, 'run_id']}/pred_{tag}.npy") for s in cm])
    bd, bci = L.boot_mean_diff(PA, PB, y, n_boot=nboot)
    row = {'model': m, 'strat_full': f_strat, 'rank': l_rank, 'strat_lora': l_strat, 'n_seed': len(cm),
           'full_mean': fu.loc[cm, col].mean(), 'lora_mean': lo.loc[cm, col].mean(), 'mean_diff': tt['mean_diff'], 'sd_diff': tt['sd_diff'],
           'ci_lo': tt['ci95_lo'], 'ci_hi': tt['ci95_hi'], 't_lower': tt['t_lower'], 't_upper': tt['t_upper'], 'p_tost': tt['p_tost'],
           'boot_diff': bd, 'boot_lo': bci[0], 'boot_hi': bci[1]}
    if wilc and len(cm) >= 10: row['p_wilcoxon'] = float(wilcoxon(d).pvalue)
    return row

def family(rows, name):
    t_ = pd.DataFrame([r_ for r_ in rows if r_]);
    if len(t_) == 0: return t_
    t_['p_tost_holm'] = L.holm(t_.p_tost.tolist()); t_['famili'] = name; t_['m_famili'] = len(t_)
    t_['keputusan'] = t_.apply(decide, axis=1); return t_

# ---- T05 agregat utama (5 seed konfirmasi)
rows = []
for (m, me, rk, s), g in main[main.seed.isin(SEEDS_CONFIRM)].groupby(['model', 'method', 'rank', 'strategy']):
    o = g[~g.diverged]
    rows.append({'model': m, 'method': me, 'rank': rk, 'strategy': s, 'n': len(g), 'n_ok': len(o), 'div_rate': 1 - len(o) / len(g),
                 'val': o.val_spearman.mean(), 'test_en': o.test_en_spearman.mean(), 'test_en_sd': o.test_en_spearman.std(ddof=1),
                 'sr_en': o.sr_en_test_spearman.mean(), 'test_id': o.test_id_spearman.mean(), 'test_id_sd': o.test_id_spearman.std(ddof=1),
                 'sr_id': o.sr_id_test_spearman.mean(), 'trainable': g.trainable_params.iloc[0], 'total': g.total_params.iloc[0]})
agg = pd.DataFrame(rows); agg['pct_trainable'] = 100 * agg.trainable / agg.total
agg['drop_rel_id'] = (agg.test_en - agg.test_id) / agg.test_en
TBL['T05_agregat_utama'] = agg

# ---- T11 divergensi (semua seed)
dv = main.groupby(['model', 'method', 'rank', 'strategy']).agg(n_run=('seed', 'count'), n_collapse=('diverged', 'sum'), n_collapse_v45=('diverged_orig', 'sum')).reset_index()
dv['laju_divergensi'] = dv.n_collapse / dv.n_run; TBL['T11_divergensi'] = dv

# ---- T06 RQ1 matched (strategi sama), T10b RQ3 matched
m1 = [pair_stats(m, s, rk, s, 'test_en_spearman', SEEDS_CONFIRM, NBOOT_MATCHED, wilc=True) for m in MODELS for s in STRATS for rk in RANKS]
TBL['T06_rq1_matched'] = family(m1, 'RQ1-matched')
m3 = [pair_stats(m, s, rk, s, 'test_id_spearman', SEEDS_CONFIRM, NBOOT_MATCHED) for m in MODELS for s in STRATS for rk in RANKS]
TBL['T10b_rq3_matched_id'] = family(m3, 'RQ3-matched')

# ---- T07 RQ1 terbaik vs terbaik (10 seed)
BEST = json.load(open(f'{STAGES}/best_pairs_v4.json')); ALLS = SEEDS_CONFIRM + SEEDS_EXTRA
b1 = [pair_stats(m, BEST[m]['full'][2], BEST[m]['lora'][1], BEST[m]['lora'][2], 'test_en_spearman', ALLS, NBOOT_BEST, wilc=True) for m in MODELS]
b3 = [pair_stats(m, BEST[m]['full'][2], BEST[m]['lora'][1], BEST[m]['lora'][2], 'test_id_spearman', ALLS, NBOOT_BEST, wilc=True) for m in MODELS]
TBL['T07_rq1_best_vs_best'] = family(b1, 'RQ1-best'); TBL['T07b_rq3_best_vs_best_id'] = family(b3, 'RQ3-best')
for k in ('T06_rq1_matched', 'T07_rq1_best_vs_best'):
    t_ = TBL[k]
    if len(t_): print(k); print(t_[['model', 'strat_full', 'rank', 'strat_lora', 'n_seed', 'mean_diff', 'ci_lo', 'ci_hi', 'p_tost', 'p_tost_holm', 'keputusan']].round(4).to_string(index=False))

In [ ]:
# ---- RQ2: efek strategi (vs S0) dan interaksi metode x strategi (5 seed konfirmasi)
def tci(d):
    n = len(d); m_ = d.mean(); se = d.std(ddof=1) / np.sqrt(n); tc = tdist.ppf(0.975, n - 1)
    return m_, m_ - tc * se, m_ + tc * se
rows = []
for m in MODELS:
    for me, rk in [('full', 0)] + [('lora', r_) for r_ in RANKS]:
        for S in ['S1', 'S2']:
            def v(s_): return okc[(okc.model == m) & (okc.method == me) & (okc['rank'] == rk) & (okc.strategy == s_)].set_index('seed').test_en_spearman
            d = (v(S) - v('S0')).dropna()
            if len(d) >= 3:
                mu, lo_, hi_ = tci(d); rows.append({'model': m, 'method': me, 'rank': rk, 'strategy': S, 'n': len(d), 'gain_vs_S0': mu, 'ci_lo': lo_, 'ci_hi': hi_, 'p': float(ttest_1samp(d, 0).pvalue)})
rq2e = pd.DataFrame(rows)
if len(rq2e): rq2e['p_holm'] = L.holm(rq2e.p.tolist()); rq2e['famili'] = 'RQ2-efek'
TBL['T09_rq2_efek_strategi'] = rq2e
rows = []
for m in MODELS:
    for S in ['S1', 'S2']:
        for rk in RANKS:
            def v(me, rk_, s_): return okc[(okc.model == m) & (okc.method == me) & (okc['rank'] == rk_) & (okc.strategy == s_)].set_index('seed').test_en_spearman
            gl = v('lora', rk, S) - v('lora', rk, 'S0'); gf = v('full', 0, S) - v('full', 0, 'S0'); dd = (gl - gf).dropna()
            if len(dd) >= 3:
                mu, lo_, hi_ = tci(dd); rows.append({'model': m, 'strategy': S, 'rank': rk, 'n': len(dd), 'gain_lora': gl.mean(), 'gain_full': gf.mean(), 'interaction': mu, 'ci_lo': lo_, 'ci_hi': hi_, 'p': float(ttest_1samp(dd, 0).pvalue)})
rq2i = pd.DataFrame(rows)
if len(rq2i): rq2i['p_holm'] = L.holm(rq2i.p.tolist()); rq2i['famili'] = 'RQ2-interaksi'
TBL['T09b_rq2_interaksi'] = rq2i
print('RQ2 efek strategi'); print(rq2e.round(4).to_string(index=False)); print('RQ2 interaksi'); print(rq2i.round(4).to_string(index=False))

# ---- RQ3: transfer ke Indonesia (Drel dengan CI bootstrap pada pasangan paralel)
rows = []
for (m, me, rk, s), g in okc.groupby(['model', 'method', 'rank', 'strategy']):
    if len(g) < 3: continue
    row = {'model': m, 'method': me, 'rank': rk, 'strategy': s, 'n_ok': len(g), 'rho_en': g.test_en_spearman.mean(), 'rho_id': g.test_id_spearman.mean(),
           'sr_en': g.sr_en_test_spearman.mean(), 'sr_id': g.sr_id_test_spearman.mean()}
    row['drop_rel'] = (row['rho_en'] - row['rho_id']) / row['rho_en']
    if ALIGNED:
        Pe = np.stack([np.load(f'{RUNS}/{r_}/pred_test_en.npy') for r_ in g.run_id]); Pi = np.stack([np.load(f'{RUNS}/{r_}/pred_test_id.npy') for r_ in g.run_id])
        _, _, _, ci = L.boot_drop_rel(Pe, EVALS['test_en']['label'].values, Pi, EVALS['test_id']['label'].values, n_boot=1000)
        row['drop_lo'], row['drop_hi'] = ci
    rows.append(row)
rq3 = pd.DataFrame(rows)
for m in MODELS:
    e_, i_ = base[f'{m}|test_en'], base[f'{m}|test_id']
    rq3 = pd.concat([rq3, pd.DataFrame([{'model': m, 'method': 'tanpa fine-tuning', 'rank': 0, 'strategy': '-', 'n_ok': 1, 'rho_en': e_, 'rho_id': i_, 'drop_rel': (e_ - i_) / e_,
                                         'sr_en': base[f'{m}|sr_en_test'], 'sr_id': base[f'{m}|sr_id_test']}])], ignore_index=True)
TBL['T10_rq3_transfer'] = rq3; print(rq3.round(4).to_string(index=False))

In [ ]:
# ---- T08 efisiensi (S0, 5 seed konfirmasi) + rasio terhadap Full
s0 = okc[okc.strategy == 'S0']
ef = s0.groupby(['model', 'method', 'rank']).agg(n=('seed', 'count'), trainable=('trainable_params', 'first'), total=('total_params', 'first'), epoch_s=('epoch_seconds', 'mean'),
        samples_per_s=('samples_per_s', 'mean'), mem_alloc_mb=('peak_gpu_mem_mb', 'mean'), mem_reserved_mb=('peak_gpu_mem_reserved_mb', 'mean'), util_gpu=('gpu_util_mean', 'mean'),
        power_w=('power_mean_w', 'mean'), energy_wh=('energy_wh', 'mean'), ckpt_mb=('checkpoint_mb', 'mean')).reset_index()
ef['pct_trainable'] = 100 * ef.trainable / ef.total
for col, nm in [('epoch_s', 'R_waktu'), ('mem_alloc_mb', 'R_memori'), ('energy_wh', 'R_energi'), ('ckpt_mb', 'R_ckpt'), ('trainable', 'R_param')]:
    ef[nm] = ef.apply(lambda r_: r_[col] / ef[(ef.model == r_.model) & (ef.method == 'full')][col].iloc[0], axis=1)
TBL['T08_efisiensi'] = ef; print(ef.round(3).to_string(index=False))

# ---- T13 ablasi (xlm-roberta-base, S0, seed ablasi)
m_ab = 'xlm-roberta-base'
a0 = df_all[(df_all.model == m_ab) & (df_all.method == 'lora') & (df_all.strategy == 'S0') & df_all.seed.isin(SEEDS_ABL) & (df_all.epochs == EPOCHS)].copy()
a0 = a0[[(abs(l_ - lr_final[f'{m_ab}|lora|{rk}|S0']) < 1e-12) for l_, rk in zip(a0.lr, a0['rank'])]]
a0['varian'] = np.where(a0.rslora, 'rsLoRA (semua linear)', np.where(a0.lora_targets == 'qv', 'LoRA q,v-saja', 'LoRA standar (semua linear)'))
a0 = a0[((a0.varian != 'LoRA standar (semua linear)') | a0['rank'].isin(sorted(set(ABL_RS_RANKS) | set(ABL_QV_RANKS))))]
abt = a0.groupby(['varian', 'rank']).apply(lambda g: pd.Series({'n': len(g), 'n_collapse': int(g.diverged.sum()), 'val': g.loc[~g.diverged, 'val_spearman'].mean(),
        'test_en': g.loc[~g.diverged, 'test_en_spearman'].mean(), 'test_en_sd': g.loc[~g.diverged, 'test_en_spearman'].std(ddof=1), 'test_id': g.loc[~g.diverged, 'test_id_spearman'].mean(),
        'trainable': g.trainable_params.iloc[0], 'epoch_s': g.epoch_seconds.mean(), 'mem_mb': g.peak_gpu_mem_mb.mean()}), include_groups=False).reset_index()
TBL['T13_ablasi'] = abt; print(abt.round(4).to_string(index=False))
TBL['T03_lr_search'] = T_LR; TBL['T04_screening'] = scr.reset_index(); TBL['T04b_uji_stabilitas'] = TBL_PROBE; TBL['T12_benchmark_gpu'] = bt
if len(fbk): TBL['T03b_lr_fallback'] = fbk
mark_done('v4_9_stats')

## Tahap 10 — Laporan komputasi dan sumber daya (log tambahan)
**Keterangan**
- *Library:* `pandas`; berkas `logs/compute_log.jsonl`, `logs/env_snapshot_*.json`, `result.json` tiap run.
- *Algoritma:* agregasi waktu dinding, jam latih, energi (kWh, Rumus 16), utilisasi/daya rata-rata, memori puncak per tahap & GPU; durasi sesi; estimasi unit = jam dinding × `UNIT_RATE`; konsumsi unit nyata dari catatan manual `L.log_units`.
- *Model:* —
- *Alasan:* biaya komputasi dan energi adalah bagian hasil RQ1; harus dilaporkan bersama jenis GPU, presisi, dan jumlah sesi.

In [ ]:
res_all = L.collect_results(RUNS); res_all['stage'] = res_all['stage'].fillna('(v3/sebelum log)')
ev = pd.read_json(f'{ROOT}/logs/compute_log.jsonl', lines=True); ev['ts'] = pd.to_datetime(ev['ts'], utc=True)
def rate_of(g): return next((v for k, v in UNIT_RATE.items() if k in str(g) and v), None)
st = res_all.groupby(['stage', 'gpu']).agg(runs=('run_id', 'count'), train_jam=('train_seconds', lambda x: x.sum() / 3600), wall_jam=('wall_seconds', lambda x: x.sum() / 3600),
        energi_kWh=('energy_wh', lambda x: x.sum() / 1000), util_gpu_mean=('gpu_util_mean', 'mean'), daya_W_mean=('power_mean_w', 'mean'), mem_puncak_mb=('peak_gpu_mem_mb', 'max')).reset_index()
st['unit_estimasi'] = [w * rate_of(g) if rate_of(g) else np.nan for w, g in zip(st.wall_jam, st.gpu)]
TBL['T14_komputasi_per_tahap'] = st; display(st.round(3))
ss = ev.groupby('session').ts.agg(['min', 'max']); ss['durasi_jam'] = (ss['max'] - ss['min']).dt.total_seconds() / 3600
uu = ev[ev.event == 'units_remaining'].sort_values('ts')
unit_used = round(uu.remaining.iloc[0] - uu.remaining.iloc[-1], 1) if len(uu) >= 2 else None
lines = ['# Ringkasan komputasi (otomatis)', f'- Jumlah run selesai: {len(res_all)}', f'- Jam dinding total: {res_all.wall_seconds.sum() / 3600:.2f} h',
         f'- Jam latih total: {res_all.train_seconds.sum() / 3600:.2f} h', f'- Energi GPU total: {res_all.energy_wh.sum() / 1000:.3f} kWh (NVML, hanya GPU)',
         f"- GPU: {sorted(res_all.gpu.unique())} | presisi: {sorted(res_all.precision.unique())}", f'- Sesi Colab: {len(ss)} | resume: {int((ev.event == "resume").sum())}',
         f'- Unit terpakai (catatan manual): {unit_used}']
open(f'{ROOT}/logs/compute_summary.md', 'w').write('\n'.join(lines)); print('\n'.join(lines)); mark_done('v4_10_compute')

## Tahap 11 — Tabel dan grafik siap naskah
**Keterangan**
- *Library:* `matplotlib`, `pandas`, `openpyxl`.
- *Algoritma:* seluruh tabel (T01–T14) diekspor ke CSV dan satu berkas Excel (`tabel_naskah.xlsx`); versi teks "rerata ± SD" untuk tabel utama; 11 gambar (PNG 200 dpi) disimpan di `figures/`. Tidak ada angka yang diketik manual: semua berasal dari tabel di atas.
- *Model:* —
- *Alasan:* menjamin angka di naskah dapat ditelusuri ke berkas keluaran dan dapat dihasilkan ulang. Pemetaan ke naskah tertera di bawah ini.

| Luaran | Isi | Bagian naskah |
|---|---|---|
| T01, T02 | Dataset; hiperparameter | Metode |
| T03, T03b, Gambar 2 | Pencarian LR per rank, *fallback* | Metode / Hasil (stabilitas) |
| T05, Gambar 3 | Agregat utama (Spearman test, SD, n) | Hasil RQ1 |
| T06, T07, Gambar 4, 11 | LoRA vs Full: matched dan terbaik-vs-terbaik (TOST, CI, Holm) | Hasil RQ1 |
| T08, T12, Gambar 5, 6 | Efisiensi (parameter, memori, waktu, energi, checkpoint), benchmark GPU | Hasil RQ1 |
| T09, T09b, Gambar 7 | Efek strategi dan interaksi | Hasil RQ2 |
| T10, T10b, T07b, Gambar 8 | Transfer ke Indonesia | Hasil RQ3 |
| T11, Gambar 10 | Laju divergensi (*collapse*) | Hasil / Pembahasan |
| T13 | Ablasi rsLoRA dan q,v-saja | Pembahasan |
| T14, Gambar 9 | Komputasi dan energi | Lampiran / Keterbatasan |

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'axes.grid': True, 'grid.alpha': .3, 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
SHORT = {'xlm-roberta-base': 'XLM-R base', 'roberta-base': 'RoBERTa base'}
MLAB = lambda me, rk: 'Full FT' if me == 'full' else f'LoRA r={rk}'
ORDER = [('full', 0)] + [('lora', r_) for r_ in RANKS]
COL = {'full': '#333333', 4: '#9ecae1', 8: '#6baed6', 16: '#3182bd', 32: '#08519c'}
SCOL = {'S0': '#4c78a8', 'S1': '#f58518', 'S2': '#54a24b'}
def ckey(me, rk): return 'full' if me == 'full' else rk
FIGLOG = []
def savefig(fig, name):
    p = f'{FIGS}/{name}.png'; fig.savefig(p, bbox_inches='tight'); plt.close(fig); FIGLOG.append(p); print('OK ', name)
def safe(fn):
    try: fn()
    except Exception as e: print('GAGAL', fn.__name__, '->', repr(e))

# ---- T01 dataset, T02 hiperparameter
TBL['T01_dataset'] = pd.DataFrame([
  {'dataset': 'STS-B (sentence-transformers/stsb)', 'bahasa': 'EN', 'peran': 'latih / validasi / uji', 'jumlah': f"{len(D['train_en'])} / {len(D['val_en'])} / {len(D['test_en'])}", 'label': 'similarity 0–1'},
  {'dataset': 'SemRel2024 (SemRel/SemRel2024, eng)', 'bahasa': 'EN', 'peran': 'uji sekunder', 'jumlah': str(len(D['sr_en_test'])), 'label': 'relatedness'},
  {'dataset': 'STS-B terjemahan (LazarusNLP/stsb_mt_id)', 'bahasa': 'ID', 'peran': 'uji transfer (tanpa latih)', 'jumlah': str(len(D['test_id'])), 'label': 'similarity 0–1 (kolom correlation)'},
  {'dataset': 'SemRel2024 (ind)', 'bahasa': 'ID', 'peran': 'uji transfer sekunder', 'jumlah': str(len(D['sr_id_test'])), 'label': 'relatedness'}])
TBL['T02_hiperparameter'] = pd.DataFrame([
  ('Model', 'xlm-roberta-base (277,45 jt param.), roberta-base (124,06 jt param.)'), ('Arsitektur', 'SBERT: encoder siamese + mean pooling + kosinus'),
  ('LoRA', 'r ∈ {4, 8, 16, 32}; α = 2r (α/r = 2); dropout 0,1; target all-linear; bias none'), ('Strategi', 'S0 MSE-kosinus; S1 S0 + LLRD 0,9; S2 CoSENT (λ = 20)'),
  ('Optimizer', 'AdamW (β = 0,9/0,999; wd 0,01); clipping 1,0; scheduler linear + warmup 10 %'), ('Epoch / batch / max_len', f'{EPOCHS} / {BATCH} / 128'),
  ('LR grid Full', str(LR_FULL)), ('LR grid LoRA (per rank)', str(LR_LORA_RANK)), ('Uji stabilitas LR', f'seed {PROBE_SEEDS} (tidak dipakai untuk hasil)'), ('Seed', f'pencarian/screening {SEED_SEARCH_SCREEN}; konfirmasi {SEEDS_CONFIRM}; tambahan {SEEDS_EXTRA}; ablasi {SEEDS_ABL}'),
  ('Kriteria collapse', f'Spearman validasi terbaik < {L.DIVERGE_SPEARMAN} atau jatuh > 0,10 dari puncak ke epoch terakhir'), ('Statistik', f'TOST δ = {DELTA}; α = {ALPHA}; Holm per famili; bootstrap {NBOOT_MATCHED}/{NBOOT_BEST}'),
  ('GPU / presisi', f"{gpu_name} / {L.pick_precision(DEVICE)}")], columns=['parameter', 'nilai'])

# ---- tabel teks "rerata ± SD"
def pm(m_, s_, d=4): return f'{m_:.{d}f} ± {s_:.{d}f}' if pd.notna(s_) else (f'{m_:.{d}f}' if pd.notna(m_) else '-')
t5 = agg.copy()
t5['Metode'] = [MLAB(me, rk) for me, rk in zip(t5.method, t5['rank'])]
t5['STS-B EN (ρ)'] = [pm(a, b) for a, b in zip(t5.test_en, t5.test_en_sd)]; t5['STS-B ID (ρ)'] = [pm(a, b) for a, b in zip(t5.test_id, t5.test_id_sd)]
t5['Param. terlatih (%)'] = t5.pct_trainable.round(2); t5['n valid/n'] = [f'{a}/{b}' for a, b in zip(t5.n_ok, t5.n)]
TBL['T05t_teks_agregat'] = t5[['model', 'Metode', 'strategy', 'n valid/n', 'STS-B EN (ρ)', 'STS-B ID (ρ)', 'Param. terlatih (%)', 'div_rate']]

# ---- Gambar 1: alur penelitian
def fig1():
    f, ax = plt.subplots(figsize=(10, 3.3)); ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 3.3)
    steps = ['Data\nSTS-B / SemRel\n(+ ID)', 'Pencarian LR\nper rank (S0)', 'Screening 30 konfig.\n+ gerbang collapse', 'Konfirmasi\n150 run (5 seed)', '10 seed:\nterbaik vs terbaik\n+ ablasi + benchmark', 'Evaluasi test\n(EN, ID) sekali', 'Statistik\nTOST, bootstrap, Holm\nRQ1–RQ3']
    w = 1.25; gap = 0.2
    for i, t in enumerate(steps):
        x = 0.1 + i * (w + gap)
        ax.add_patch(FancyBboxPatch((x, 1.0), w, 1.3, boxstyle='round,pad=0.03', fc='#e8f0fa', ec='#3182bd'))
        ax.text(x + w / 2, 1.65, t, ha='center', va='center', fontsize=7.2)
        if i < len(steps) - 1: ax.annotate('', xy=(x + w + gap - 0.02, 1.65), xytext=(x + w + 0.02, 1.65), arrowprops=dict(arrowstyle='->', color='#555'))
    ax.text(5, 0.45, 'Seluruh run: checkpoint per epoch di Google Drive • validasi untuk semua pemilihan • test dipakai sekali', ha='center', fontsize=8, color='#444')
    savefig(f, 'G01_alur_penelitian')
safe(fig1)

# ---- Gambar 2: sensitivitas LR (collapse)
def fig2():
    f, axs = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    for ax, m in zip(axs, MODELS):
        for me, rk in ORDER:
            g = T_LR[(T_LR.model == m) & (T_LR.method == me) & (T_LR['rank'] == rk)].sort_values('lr')
            ax.plot(g.lr, g.val_spearman, '-o' if me == 'lora' else '--s', color=COL[ckey(me, rk)], label=MLAB(me, rk), ms=4)
            c_ = g[g.diverged]; ax.scatter(c_.lr, c_.val_spearman, marker='x', s=70, color='red', zorder=5)
        ax.axhline(L.DIVERGE_SPEARMAN, color='red', ls=':', lw=1); ax.set_xscale('log'); ax.set_title(SHORT[m]); ax.set_xlabel('learning rate (skala log)')
    axs[0].set_ylabel('Spearman validasi (S0, seed 42)'); axs[0].legend(fontsize=7, title='✕ = collapse', title_fontsize=7)
    savefig(f, 'G02_sensitivitas_LR')
safe(fig2)

# ---- Gambar 3: Spearman test per metode x strategi
def fig3():
    f, axs = plt.subplots(1, 2, figsize=(10.5, 3.8), sharey=False)
    for ax, m in zip(axs, MODELS):
        xs = np.arange(len(ORDER)); w = 0.26
        for j, S in enumerate(STRATS):
            mu, sd = [], []
            for me, rk in ORDER:
                r_ = agg[(agg.model == m) & (agg.method == me) & (agg['rank'] == rk) & (agg.strategy == S)]
                mu.append(r_.test_en.iloc[0] if len(r_) else np.nan); sd.append(r_.test_en_sd.iloc[0] if len(r_) else np.nan)
            ax.bar(xs + (j - 1) * w, mu, w, yerr=np.nan_to_num(sd), color=SCOL[S], label=S, capsize=2, error_kw=dict(lw=.8))
        lo_ = np.nanmin(agg[agg.model == m].test_en); ax.set_ylim(max(0, lo_ - 0.03), np.nanmax(agg[agg.model == m].test_en) + 0.012)
        ax.axhline(base[f'{m}|test_en'], color='gray', ls=':', lw=1) if base[f'{m}|test_en'] > ax.get_ylim()[0] else None
        ax.set_xticks(xs); ax.set_xticklabels([MLAB(me, rk) for me, rk in ORDER]); ax.set_title(SHORT[m]); ax.set_ylabel('Spearman STS-B test (EN)')
    axs[0].legend(title='strategi', fontsize=7); savefig(f, 'G03_spearman_test_metode_strategi')
safe(fig3)

# ---- Gambar 4: forest plot Delta (LoRA - Full) dengan pita ekuivalensi
def fig4():
    f, axs = plt.subplots(1, 2, figsize=(12, 6.5), sharex=True, gridspec_kw={'wspace': 0.55})
    t6, t7 = TBL['T06_rq1_matched'], TBL['T07_rq1_best_vs_best']
    for ax, m in zip(axs, MODELS):
        rows = []
        for _, r_ in t7[t7.model == m].iterrows(): rows.append((f"TERBAIK r={int(r_['rank'])} {r_.strat_lora} (n={r_.n_seed})", r_, '#d62728'))
        for S in STRATS:
            for rk in RANKS:
                g = t6[(t6.model == m) & (t6.strat_lora == S) & (t6['rank'] == rk)]
                if len(g): rows.append((f'{S} | r={rk}', g.iloc[0], SCOL[S]))
        ys = np.arange(len(rows))[::-1]
        ax.axvspan(-DELTA, DELTA, color='#2ca02c', alpha=.12, label=f'zona ekuivalensi ±{DELTA}'); ax.axvline(0, color='k', lw=.8)
        for y_, (lab, r_, c_) in zip(ys, rows):
            ax.errorbar(r_.mean_diff, y_, xerr=[[r_.mean_diff - r_.ci_lo], [r_.ci_hi - r_.mean_diff]], fmt='o', color=c_, capsize=2, ms=4)
        ax.set_yticks(ys); ax.set_yticklabels([r_[0] for r_ in rows], fontsize=7); ax.set_title(SHORT[m]); ax.set_xlabel('Δ Spearman = LoRA − Full (CI-t 95 %)')
    axs[0].legend(fontsize=7, loc='lower left'); savefig(f, 'G04_forest_ekuivalensi')
safe(fig4)

# ---- Gambar 5: rasio efisiensi terhadap Full
def fig5():
    f, axs = plt.subplots(1, 4, figsize=(12, 3.2))
    for ax, (col, ttl) in zip(axs, [('R_waktu', 'waktu / epoch'), ('R_memori', 'memori GPU puncak'), ('R_energi', 'energi / run'), ('R_ckpt', 'ukuran checkpoint')]):
        for j, m in enumerate(MODELS):
            g = ef[(ef.model == m) & (ef.method == 'lora')].sort_values('rank')
            ax.bar(np.arange(len(g)) + (j - .5) * .38, g[col], .38, label=SHORT[m], color=['#6baed6', '#fd8d3c'][j])
        ax.axhline(1, color='k', lw=.8, ls='--'); ax.set_xticks(range(len(RANKS))); ax.set_xticklabels([f'r={r_}' for r_ in RANKS]); ax.set_title(f'{ttl}\n(rasio LoRA/Full)')
        if col == 'R_ckpt': ax.set_yscale('log')
    axs[0].legend(fontsize=7); savefig(f, 'G05_rasio_efisiensi')
safe(fig5)

# ---- Gambar 6: trade-off akurasi vs biaya (S0)
def fig6():
    f, axs = plt.subplots(1, 2, figsize=(10, 3.8))
    for m, mk in zip(MODELS, ['o', 's']):
        for _, r_ in ef[ef.model == m].iterrows():
            a_ = agg[(agg.model == m) & (agg.method == r_.method) & (agg['rank'] == r_['rank']) & (agg.strategy == 'S0')]
            if not len(a_): continue
            y_ = a_.test_en.iloc[0]
            for ax, x_ in zip(axs, [r_.mem_alloc_mb / 1024, r_.pct_trainable]):
                ax.scatter(x_, y_, marker=mk, color=COL[ckey(r_.method, r_['rank'])], s=55, edgecolor='k', lw=.4); ax.annotate(MLAB(r_.method, r_['rank']), (x_, y_), fontsize=6, xytext=(3, 3), textcoords='offset points')
    axs[0].set_xlabel('memori GPU puncak (GB)'); axs[1].set_xlabel('% parameter terlatih (skala log)'); axs[1].set_xscale('log')
    for ax in axs: ax.set_ylabel('Spearman STS-B test (S0)')
    axs[0].scatter([], [], marker='o', color='gray', label=SHORT[MODELS[0]]); axs[0].scatter([], [], marker='s', color='gray', label=SHORT[MODELS[1]]); axs[0].legend(fontsize=7)
    savefig(f, 'G06_tradeoff_akurasi_biaya')
safe(fig6)

# ---- Gambar 7: efek strategi dan interaksi
def fig7():
    f, axs = plt.subplots(1, 2, figsize=(10.5, 3.6), sharey=True)
    for ax, m in zip(axs, MODELS):
        for j, S in enumerate(['S1', 'S2']):
            g = rq2e[(rq2e.model == m) & (rq2e.strategy == S)].copy(); g['k'] = [MLAB(a, b) for a, b in zip(g.method, g['rank'])]
            g = g.set_index('k').reindex([MLAB(me, rk) for me, rk in ORDER]).dropna(subset=['gain_vs_S0'])
            xs = [ [MLAB(me, rk) for me, rk in ORDER].index(k) for k in g.index ]
            ax.errorbar(np.array(xs) + (j - .5) * .2, g.gain_vs_S0, yerr=[g.gain_vs_S0 - g.ci_lo, g.ci_hi - g.gain_vs_S0], fmt='o', color=SCOL[S], capsize=2, label=S)
        ax.axhline(0, color='k', lw=.8); ax.set_xticks(range(len(ORDER))); ax.set_xticklabels([MLAB(me, rk) for me, rk in ORDER], fontsize=8); ax.set_title(SHORT[m])
    axs[0].set_ylabel('Δ Spearman test vs S0 (CI-t 95 %)'); axs[0].legend(fontsize=7); savefig(f, 'G07_efek_strategi')
safe(fig7)

# ---- Gambar 8: transfer EN -> ID
def fig8():
    f, axs = plt.subplots(1, 2, figsize=(10.5, 3.8))
    r3 = rq3[rq3.method != 'tanpa fine-tuning']
    for _, r_ in r3.iterrows():
        axs[0].scatter(r_.rho_en, r_.rho_id, color=COL[ckey(r_.method, r_['rank'])], marker='o' if r_.model == MODELS[0] else 's', edgecolor='k', lw=.4, s=40)
    lim = [min(r3.rho_id.min(), r3.rho_en.min()) - .01, r3.rho_en.max() + .01]; axs[0].plot(lim, lim, 'k--', lw=.8); axs[0].set_xlabel('ρ STS-B EN'); axs[0].set_ylabel('ρ STS-B ID (tanpa pelatihan ID)')
    s0_ = r3[r3.strategy == 'S0']; xs = np.arange(len(ORDER));
    for j, m in enumerate(MODELS):
        v = [s0_[(s0_.model == m) & (s0_.method == me) & (s0_['rank'] == rk)].drop_rel.mean() * 100 for me, rk in ORDER]
        axs[1].bar(xs + (j - .5) * .38, v, .38, label=SHORT[m])
    axs[1].set_xticks(xs); axs[1].set_xticklabels([MLAB(me, rk) for me, rk in ORDER], fontsize=8); axs[1].set_ylabel('penurunan relatif Δrel (%) — S0'); axs[1].legend(fontsize=7)
    savefig(f, 'G08_transfer_EN_ID')
safe(fig8)

# ---- Gambar 9: komputasi dan energi
def fig9():
    f, axs = plt.subplots(1, 2, figsize=(10, 3.5))
    g = st.groupby('stage').wall_jam.sum().sort_values(); axs[0].barh(g.index, g.values, color='#6baed6'); axs[0].set_xlabel('jam dinding GPU')
    r_ = res_all[res_all.stage.isin(['confirm', 'confirm_extra'])] if res_all.stage.isin(['confirm', 'confirm_extra']).any() else res_all
    data = [r_[(r_.model == m) & (r_.method == me) & (r_['rank'] == rk)].energy_wh.dropna() for m in MODELS for me, rk in ORDER]
    labs = [f'{SHORT[m][:4]}\n{MLAB(me, rk).replace("LoRA ", "")}' for m in MODELS for me, rk in ORDER]
    axs[1].boxplot([d_ if len(d_) else [np.nan] for d_ in data], tick_labels=labs); axs[1].set_ylabel('energi GPU per run (Wh)'); axs[1].tick_params(axis='x', labelsize=6)
    savefig(f, 'G09_komputasi_energi')
safe(fig9)

# ---- Gambar 10: peta panas laju divergensi
def fig10():
    f, axs = plt.subplots(1, 2, figsize=(8.5, 3.6))
    for ax, m in zip(axs, MODELS):
        M = np.array([[dv[(dv.model == m) & (dv.method == me) & (dv['rank'] == rk) & (dv.strategy == S)].laju_divergensi.mean() for S in STRATS] for me, rk in ORDER])
        im = ax.imshow(M, vmin=0, vmax=1, cmap='Reds', aspect='auto'); ax.set_xticks(range(3)); ax.set_xticklabels(STRATS); ax.set_yticks(range(len(ORDER))); ax.set_yticklabels([MLAB(me, rk) for me, rk in ORDER])
        for i in range(M.shape[0]):
            for j in range(M.shape[1]): ax.text(j, i, f'{M[i, j]:.0%}' if not np.isnan(M[i, j]) else '-', ha='center', va='center', fontsize=8)
        ax.set_title(SHORT[m]); ax.grid(False)
    f.suptitle('Laju divergensi (collapse) pada konfirmasi', fontsize=9); savefig(f, 'G10_laju_divergensi')
safe(fig10)

# ---- Gambar 11: Spearman per seed, Full vs LoRA terbaik (10 seed)
def fig11():
    f, axs = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=False)
    for ax, m in zip(axs, MODELS):
        bf, bl = BEST[m]['full'], BEST[m]['lora']
        fu = okm[(okm.model == m) & (okm.method == 'full') & (okm.strategy == bf[2])].set_index('seed').test_en_spearman
        lo = okm[(okm.model == m) & (okm.method == 'lora') & (okm['rank'] == bl[1]) & (okm.strategy == bl[2])].set_index('seed').test_en_spearman
        cm = sorted(set(fu.index) & set(lo.index))
        for s_ in cm: ax.plot([0, 1], [fu[s_], lo[s_]], '-', color='#bbb', lw=.8)
        ax.scatter([0] * len(cm), fu[cm], color=COL['full'], zorder=3); ax.scatter([1] * len(cm), lo[cm], color=COL[bl[1]], zorder=3)
        ax.set_xticks([0, 1]); ax.set_xticklabels([f'Full ({bf[2]})', f'LoRA r={bl[1]} ({bl[2]})']); ax.set_title(f'{SHORT[m]} — {len(cm)} seed berpasangan'); ax.set_ylabel('Spearman STS-B test (EN)')
    savefig(f, 'G11_per_seed_terbaik')
safe(fig11)

In [ ]:
ALG_TEXT = '\n## Lampiran A — Rumus matematika dan algoritma (siap disalin ke naskah)\n\n### A.1 Rumus\n\n**(1) Representasi kalimat dan kemiripan (SBERT; Reimers & Gurevych, 2019).** Untuk kalimat $a$ dengan keluaran encoder $h_t\\in\\mathbb{R}^{d}$ dan mask $m_t\\in\\{0,1\\}$:\n\n$$u=\\frac{\\sum_t m_t h_t}{\\sum_t m_t},\\qquad s(a,b)=\\cos(u,v)=\\frac{u^{\\top}v}{\\lVert u\\rVert\\,\\lVert v\\rVert}$$\n\n**(2) Fungsi kerugian S0 (MSE kosinus).** Untuk *batch* berukuran $B$ dan skor acuan $y_i\\in[0,1]$:\n\n$$\\mathcal{L}_{\\mathrm{MSE}}=\\frac{1}{B}\\sum_{i=1}^{B}\\bigl(s_i-y_i\\bigr)^2$$\n\n**(3) Fungsi kerugian S2 (CoSENT; Su, 2022)** dengan faktor skala $\\lambda=20$:\n\n$$\\mathcal{L}_{\\mathrm{CoSENT}}=\\log\\Bigl(1+\\sum_{(i,j):\\,y_i>y_j}\\exp\\bigl(\\lambda\\,(s_j-s_i)\\bigr)\\Bigr)$$\n\n**(4) LoRA (Hu dkk., 2022).** Bobot beku $W_0\\in\\mathbb{R}^{d\\times k}$ diperbarui secara *low-rank*; $B\\in\\mathbb{R}^{d\\times r}$, $A\\in\\mathbb{R}^{r\\times k}$, $r\\ll\\min(d,k)$:\n\n$$W=W_0+\\frac{\\alpha}{r}BA,\\qquad \\#\\theta_{\\mathrm{LoRA}}=\\sum_{\\text{lapisan}} r\\,(d+k),\\qquad \\alpha=2r\\;\\Rightarrow\\;\\frac{\\alpha}{r}=2$$\n\n**(5) rsLoRA (Kalajdzievski, 2023)** mengganti skala menjadi $\\gamma_r=\\alpha/\\sqrt{r}$ (pada penelitian ini $\\alpha=2\\sqrt{8}$ sehingga $\\gamma_8=2$, sama dengan LoRA standar pada $r=8$).\n\n**(6) Layer-wise LR decay, S1 (Howard & Ruder, 2018)** untuk lapisan $\\ell\\in\\{0,\\dots,L-1\\}$ dan faktor $\\rho=0{,}9$:\n\n$$\\eta_\\ell=\\eta\\,\\rho^{\\,L-1-\\ell}$$\n\n**(7) Optimisasi.** AdamW (Loshchilov & Hutter, 2019), $\\beta=(0{,}9;0{,}999)$, *weight decay* $0{,}01$, *clipping* norma gradien $1{,}0$; laju belajar linear dengan *warmup* 10 %:\n\n$$\\eta_t=\\eta\\cdot\\begin{cases} t/T_w, & t<T_w\\\\[2pt] (T-t)/(T-T_w), & t\\ge T_w\\end{cases}$$\n\n**(8) Metrik.** Korelasi Spearman (metrik utama) adalah korelasi Pearson antar-peringkat; tanpa nilai kembar $\\rho=1-\\dfrac{6\\sum_i d_i^2}{n(n^2-1)}$ dengan $d_i$ selisih peringkat.\n\n**(9) Kriteria *collapse* dan laju divergensi.** Satu *run* disebut *collapse* bila $\\max_{e}\\rho^{\\mathrm{val}}_{e}<0{,}50$ (di bawah baseline tanpa fine-tuning). Laju divergensi sebuah konfigurasi: $\\mathrm{DR}=n_{\\mathrm{collapse}}/n_{\\mathrm{run}}$; *run collapse* dikeluarkan dari rerata dan dilaporkan melalui DR.\n\n**(10) Selisih berpasangan per seed.** Untuk seed $k$ dan metrik $\\rho$: $d_k=\\rho^{\\mathrm{LoRA}}_k-\\rho^{\\mathrm{Full}}_k$; rerata $\\bar d$, simpangan baku $s_d$, $n$ pasangan seed.\n\n**(11) Uji ekuivalensi TOST (Lakens, 2017)** dengan margin $\\delta=0{,}01$:\n\n$$t_L=\\frac{\\bar d+\\delta}{s_d/\\sqrt n},\\quad t_U=\\frac{\\bar d-\\delta}{s_d/\\sqrt n},\\quad p_{\\mathrm{TOST}}=\\max\\bigl(P(T_{n-1}\\ge t_L),\\;P(T_{n-1}\\le t_U)\\bigr)$$\n\nLoRA dinyatakan *ekuivalen* dengan full fine-tuning bila $p_{\\mathrm{TOST}}^{\\mathrm{Holm}}<0{,}05$. Interval kepercayaan 95 %: $\\bar d\\pm t_{0{,}975,\\,n-1}\\,s_d/\\sqrt n$.\n\n**(12) *Bootstrap* berpasangan atas pasangan kalimat uji.** Pada $b=1,\\dots,B_{\\mathrm{boot}}$, indeks kalimat diambil dengan pengembalian ($n=1379$); statistik $\\Delta^{(b)}=\\overline{\\rho}^{\\,(b)}_{\\mathrm{LoRA}}-\\overline{\\rho}^{\\,(b)}_{\\mathrm{Full}}$ (rerata lintas seed); CI 95 % = persentil ke-2,5 dan 97,5.\n\n**(13) Koreksi Holm (1979).** Untuk $m$ pengujian dalam satu famili, p-value terurut $p_{(1)}\\le\\dots\\le p_{(m)}$:\n\n$$\\tilde p_{(i)}=\\min\\Bigl(1,\\;\\max_{j\\le i}\\,(m-j+1)\\,p_{(j)}\\Bigr)$$\n\n**(14) Interaksi metode × strategi (RQ2).** Dengan $g^{M}_{S,k}=\\rho^{M}_{S,k}-\\rho^{M}_{S0,k}$ ($M\\in\\{\\mathrm{LoRA},\\mathrm{Full}\\}$): $I_S=\\overline{g^{\\mathrm{LoRA}}_S-g^{\\mathrm{Full}}_S}$; diuji dengan uji-$t$ satu sampel terhadap 0.\n\n**(15) Penurunan relatif transfer (RQ3).**\n\n$$\\Delta_{\\mathrm{rel}}=\\frac{\\rho_{\\mathrm{EN}}-\\rho_{\\mathrm{ID}}}{\\rho_{\\mathrm{EN}}}$$\n\ndengan CI *bootstrap* atas pasangan paralel EN–ID (indeks sama).\n\n**(16) Sumber daya.** Energi GPU $E=\\int_0^{T}P(t)\\,dt\\approx\\dfrac{1}{3600}\\sum_k P_k\\,\\Delta t_k$ (Wh); rasio efisiensi terhadap full fine-tuning $R_X=X_{\\mathrm{LoRA}}/X_{\\mathrm{Full}}$ untuk $X\\in\\{$waktu per epoch, memori puncak, energi, ukuran *checkpoint*, parameter terlatih$\\}$.\n\n### A.2 Algoritma\n\n**Algoritma 1 — Protokol eksperimen keseluruhan**\n```\nInput : model M ∈ {XLM-R base, RoBERTa base}, metode {Full, LoRA r∈{4,8,16,32}}, strategi {S0,S1,S2}\n1  Muat STS-B (train/val/test), SemRel2024 EN & ID, STS-B terjemahan ID; verifikasi pasangan EN–ID paralel\n2  untuk setiap (M, metode, rank): jalankan Algoritma 3 pada S0, seed 42  → LR awal η0\n3  untuk setiap (M, metode, rank, strategi): latih seed 42 dengan η0; bila collapse → turunkan LR satu langkah grid (Algoritma 3)\n4  untuk setiap konfigurasi (30) dan seed k∈{1..5}: Algoritma 2  → bobot terbaik berdasarkan Spearman validasi\n5  pilih Full terbaik dan LoRA terbaik per model (rerata Spearman validasi, laju divergensi ≤ 20 %)\n6  tambah seed {6..10} untuk dua konfigurasi terpilih → 10 seed\n7  ablasi (rsLoRA r∈{16,32}; q,v-saja r∈{8,16}) dan benchmark GPU\n8  evaluasi SEKALI pada STS-B test (EN), SemRel EN, STS-B test (ID), SemRel ID\n9  Algoritma 4 untuk RQ1, RQ2 (interaksi), RQ3 (Holm per famili)\nOutput: tabel, grafik, log sumber daya\n```\n\n**Algoritma 2 — Pelatihan SBERT (Full atau LoRA) dengan checkpoint**\n```\nInput : cfg (model, metode, rank, strategi, η, seed), data latih/validasi\n1  id ← hash(cfg); bila result.json ada → lewati; bila checkpoint ada → lanjutkan dari epoch terakhir\n2  inisialisasi encoder; bila LoRA: bekukan W0, pasang (A,B) pada semua lapisan linear\n3  untuk epoch e = 1..4:\n4      urutan data ← permutasi deterministik(seed·1000 + e)\n5      untuk setiap batch (B=32): u,v ← SBERT(a), SBERT(b); L ← MSE (S0,S1) atau CoSENT (S2)\n6          backprop; clip norma gradien 1,0; AdamW (LLRD pada S1); scheduler.step()\n7      ρ_val ← Spearman(prediksi validasi); simpan bobot terlatih, optimizer, RNG, riwayat (atomik)\n8  bobot terbaik ← epoch dengan ρ_val tertinggi; diverged ← (max ρ_val < 0,50)\n9  catat waktu, memori puncak, daya/energi (NVML), ukuran checkpoint\n```\n\n**Algoritma 3 — Pencarian LR dengan deteksi collapse**\n```\nInput : grid G (urut menurun), konfigurasi (M, metode, rank, strategi)\n1  untuk η ∈ G: latih (Algoritma 2, seed 42); simpan ρ_val dan flag diverged\n2  η* ← argmax ρ_val di antara run yang TIDAK diverged\n3  (saat screening strategi S1/S2) bila run dengan η* diverged: η* ← LR berikutnya yang lebih kecil di G; ulangi\n4  bila G habis dan masih diverged → hentikan (jeda otomatis) dan laporkan\nOutput: LR per (model, metode, rank, strategi); catatan fallback\n```\n\n**Algoritma 4 — Perbandingan berpasangan LoRA vs Full**\n```\nInput : prediksi uji per seed, seed yang lolos (tidak diverged) pada kedua metode\n1  d_k ← ρ_LoRA,k − ρ_Full,k untuk seed bersama (n ≥ 3)\n2  hitung d̄, s_d, CI-t 95 %, p_TOST (δ = 0,01)\n3  bootstrap kalimat uji (B_boot) pada rerata prediksi lintas seed → CI 95 % Δρ\n4  (hanya bila n ≥ 10) uji Wilcoxon signed-rank\n5  koreksi Holm dalam satu famili pengujian\n6  keputusan: ekuivalen bila p_Holm < 0,05; "LoRA lebih buruk/baik" bila seluruh CI-t di luar [−δ, +δ]\n```\n'

## Tahap 11a — Tabel per run dan deteksi pencilan terdegradasi (lampiran, tanpa GPU)
**Keterangan**
- *Library:* `pandas`, `sts_lora_lib.collect_results`.
- *Algoritma:* menyimpan SETIAP run (seed, LR, Spearman validasi per epoch, Spearman test EN/ID, penanda *collapse* kriteria v4.5 dan v4.6). Pencilan terdegradasi = run yang tidak ditandai *collapse* tetapi Spearman test EN-nya lebih rendah dari median konfigurasinya sebesar > 0,05 (diagnostik, TIDAK dipakai untuk mengeluarkan run).
- *Model matematika:* $\text{pencilan}_i = \mathbb{1}[\operatorname{median}_{j \in cfg}(\rho_j) - \rho_i > 0{,}05]$.
- *Alasan:* kriteria *collapse* hanya melihat lintasan validasi; run yang pulih sebagian dapat lolos dan memengaruhi rerata. Tabel ini memperlihatkannya secara terbuka (lampiran naskah) dan dipakai untuk analisis sensitivitas "dengan dan tanpa pencilan".

In [ ]:
import glob
hist_ = {}
for f_ in glob.glob(f'{RUNS}/*/result.json'):
    r_ = json.load(open(f_)); hist_[r_['run_id']] = r_.get('history') or []
pr = L.collect_results(RUNS)
pr['val_per_epoch'] = pr.run_id.map(lambda k: ' / '.join(f"{h['val_spearman']:.3f}" for h in hist_.get(k, [])))
pr['val_last'] = pr.run_id.map(lambda k: hist_[k][-1]['val_spearman'] if hist_.get(k) else np.nan)
cfgk = ['model', 'method', 'rank', 'strategy', 'lr']
ok_ = pr[~pr.diverged].copy()
med_ = ok_.groupby(cfgk).test_en_spearman.transform('median')
pr['median_cfg_test_en'] = ok_.assign(m=med_).m.reindex(pr.index)
pr['pencilan_turun'] = (~pr.diverged) & ((pr.median_cfg_test_en - pr.test_en_spearman) > 0.05)
cols_ = ['model', 'method', 'rank', 'strategy', 'lr', 'seed', 'stage', 'diverged_orig', 'diverged', 'pencilan_turun', 'val_min', 'val_last', 'val_spearman',
         'test_en_spearman', 'test_id_spearman', 'val_per_epoch']
TBL['T15_per_run'] = pr[[c for c in cols_ if c in pr.columns]].sort_values(['model', 'method', 'rank', 'strategy', 'seed'])
pen = TBL['T15_per_run'][TBL['T15_per_run'].pencilan_turun]
TBL['T15b_pencilan_terdegradasi'] = pen
TBL['T16_audit_bobot_terbaik'] = aud.drop(columns=['run_id'])
TBL['T16b_run_dilatih_ulang'] = pd.DataFrame({'run_id_dilatih_ulang': repaired})
print('Run per tabel T15:', len(pr), '| collapse v4.5:', int(pr.diverged_orig.sum()), '| collapse v4.6:', int(pr.diverged.sum()), '| pencilan terdegradasi:', len(pen))
print(pen.to_string(index=False) if len(pen) else '(tidak ada)')

In [ ]:
# ---- Ekspor: CSV, Excel, tabel teks (Markdown), dan paket ZIP
import zipfile
for k, t_ in TBL.items():
    if isinstance(t_, pd.DataFrame): t_.to_csv(f'{TABLES}/{k}.csv', index=False)
try:
    with pd.ExcelWriter(f'{TABLES}/tabel_naskah.xlsx') as xw:
        for k, t_ in TBL.items():
            if isinstance(t_, pd.DataFrame): t_.round(5).to_excel(xw, sheet_name=k[:31], index=False)
except Exception as e: print('Excel gagal:', e)
def to_md(t_):
    try: return t_.to_markdown(index=False, floatfmt='.4f')
    except Exception: return t_.to_string(index=False)
with open(f'{TABLES}/tabel_naskah.md', 'w') as fh:
    for k in ['T01_dataset', 'T02_hiperparameter', 'T05t_teks_agregat', 'T07_rq1_best_vs_best', 'T08_efisiensi', 'T10_rq3_transfer', 'T11_divergensi', 'T13_ablasi']:
        if k in TBL: fh.write(f'\n### {k}\n\n' + to_md(TBL[k].round(4)) + '\n')
open(f'{TABLES}/algoritma_dan_rumus.md', 'w').write(ALG_TEXT)
zp = f'{ROOT}/luaran_naskah_v4b.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in Path(TABLES).glob('*'):
        if p.suffix in ('.csv', '.xlsx', '.md', '.json'): z.write(p, f'tabel_v4/{p.name}')
    for p in Path(FIGS).glob('*.png'): z.write(p, f'figures/{p.name}')
    for p in [Path(f'{ROOT}/logs/compute_summary.md'), Path(f'{STAGES}/lr_final_v4.json'), Path(f'{STAGES}/lr_search_v4.json'), Path(f'{STAGES}/best_pairs_v4.json')]:
        if p.exists(): z.write(p, f'meta/{p.name}')
print('Tabel:', len(TBL), '| Gambar:', len(FIGLOG), '| paket:', zp, f'({os.path.getsize(zp) / 1e6:.1f} MB)'); mark_done('v4_11_outputs')
from IPython.display import Image, display as _d
for p in sorted(FIGLOG)[:3]: _d(Image(p))

## Lampiran A — Rumus matematika dan algoritma (siap disalin ke naskah)

### A.1 Rumus

**(1) Representasi kalimat dan kemiripan (SBERT; Reimers & Gurevych, 2019).** Untuk kalimat $a$ dengan keluaran encoder $h_t\in\mathbb{R}^{d}$ dan mask $m_t\in\{0,1\}$:

$$u=\frac{\sum_t m_t h_t}{\sum_t m_t},\qquad s(a,b)=\cos(u,v)=\frac{u^{\top}v}{\lVert u\rVert\,\lVert v\rVert}$$

**(2) Fungsi kerugian S0 (MSE kosinus).** Untuk *batch* berukuran $B$ dan skor acuan $y_i\in[0,1]$:

$$\mathcal{L}_{\mathrm{MSE}}=\frac{1}{B}\sum_{i=1}^{B}\bigl(s_i-y_i\bigr)^2$$

**(3) Fungsi kerugian S2 (CoSENT; Su, 2022)** dengan faktor skala $\lambda=20$:

$$\mathcal{L}_{\mathrm{CoSENT}}=\log\Bigl(1+\sum_{(i,j):\,y_i>y_j}\exp\bigl(\lambda\,(s_j-s_i)\bigr)\Bigr)$$

**(4) LoRA (Hu dkk., 2022).** Bobot beku $W_0\in\mathbb{R}^{d\times k}$ diperbarui secara *low-rank*; $B\in\mathbb{R}^{d\times r}$, $A\in\mathbb{R}^{r\times k}$, $r\ll\min(d,k)$:

$$W=W_0+\frac{\alpha}{r}BA,\qquad \#\theta_{\mathrm{LoRA}}=\sum_{\text{lapisan}} r\,(d+k),\qquad \alpha=2r\;\Rightarrow\;\frac{\alpha}{r}=2$$

**(5) rsLoRA (Kalajdzievski, 2023)** mengganti skala menjadi $\gamma_r=\alpha/\sqrt{r}$ (pada penelitian ini $\alpha=2\sqrt{8}$ sehingga $\gamma_8=2$, sama dengan LoRA standar pada $r=8$).

**(6) Layer-wise LR decay, S1 (Howard & Ruder, 2018)** untuk lapisan $\ell\in\{0,\dots,L-1\}$ dan faktor $\rho=0{,}9$:

$$\eta_\ell=\eta\,\rho^{\,L-1-\ell}$$

**(7) Optimisasi.** AdamW (Loshchilov & Hutter, 2019), $\beta=(0{,}9;0{,}999)$, *weight decay* $0{,}01$, *clipping* norma gradien $1{,}0$; laju belajar linear dengan *warmup* 10 %:

$$\eta_t=\eta\cdot\begin{cases} t/T_w, & t<T_w\\[2pt] (T-t)/(T-T_w), & t\ge T_w\end{cases}$$

**(8) Metrik.** Korelasi Spearman (metrik utama) adalah korelasi Pearson antar-peringkat; tanpa nilai kembar $\rho=1-\dfrac{6\sum_i d_i^2}{n(n^2-1)}$ dengan $d_i$ selisih peringkat.

**(9) Kriteria *collapse* dan laju divergensi.** Satu *run* disebut *collapse* bila $\max_{e}\rho^{\mathrm{val}}_{e}<0{,}50$ (di bawah baseline tanpa fine-tuning). Laju divergensi sebuah konfigurasi: $\mathrm{DR}=n_{\mathrm{collapse}}/n_{\mathrm{run}}$; *run collapse* dikeluarkan dari rerata dan dilaporkan melalui DR.

**(10) Selisih berpasangan per seed.** Untuk seed $k$ dan metrik $\rho$: $d_k=\rho^{\mathrm{LoRA}}_k-\rho^{\mathrm{Full}}_k$; rerata $\bar d$, simpangan baku $s_d$, $n$ pasangan seed.

**(11) Uji ekuivalensi TOST (Lakens, 2017)** dengan margin $\delta=0{,}01$:

$$t_L=\frac{\bar d+\delta}{s_d/\sqrt n},\quad t_U=\frac{\bar d-\delta}{s_d/\sqrt n},\quad p_{\mathrm{TOST}}=\max\bigl(P(T_{n-1}\ge t_L),\;P(T_{n-1}\le t_U)\bigr)$$

LoRA dinyatakan *ekuivalen* dengan full fine-tuning bila $p_{\mathrm{TOST}}^{\mathrm{Holm}}<0{,}05$. Interval kepercayaan 95 %: $\bar d\pm t_{0{,}975,\,n-1}\,s_d/\sqrt n$.

**(12) *Bootstrap* berpasangan atas pasangan kalimat uji.** Pada $b=1,\dots,B_{\mathrm{boot}}$, indeks kalimat diambil dengan pengembalian ($n=1379$); statistik $\Delta^{(b)}=\overline{\rho}^{\,(b)}_{\mathrm{LoRA}}-\overline{\rho}^{\,(b)}_{\mathrm{Full}}$ (rerata lintas seed); CI 95 % = persentil ke-2,5 dan 97,5.

**(13) Koreksi Holm (1979).** Untuk $m$ pengujian dalam satu famili, p-value terurut $p_{(1)}\le\dots\le p_{(m)}$:

$$\tilde p_{(i)}=\min\Bigl(1,\;\max_{j\le i}\,(m-j+1)\,p_{(j)}\Bigr)$$

**(14) Interaksi metode × strategi (RQ2).** Dengan $g^{M}_{S,k}=\rho^{M}_{S,k}-\rho^{M}_{S0,k}$ ($M\in\{\mathrm{LoRA},\mathrm{Full}\}$): $I_S=\overline{g^{\mathrm{LoRA}}_S-g^{\mathrm{Full}}_S}$; diuji dengan uji-$t$ satu sampel terhadap 0.

**(15) Penurunan relatif transfer (RQ3).**

$$\Delta_{\mathrm{rel}}=\frac{\rho_{\mathrm{EN}}-\rho_{\mathrm{ID}}}{\rho_{\mathrm{EN}}}$$

dengan CI *bootstrap* atas pasangan paralel EN–ID (indeks sama).

**(16) Sumber daya.** Energi GPU $E=\int_0^{T}P(t)\,dt\approx\dfrac{1}{3600}\sum_k P_k\,\Delta t_k$ (Wh); rasio efisiensi terhadap full fine-tuning $R_X=X_{\mathrm{LoRA}}/X_{\mathrm{Full}}$ untuk $X\in\{$waktu per epoch, memori puncak, energi, ukuran *checkpoint*, parameter terlatih$\}$.

### A.2 Algoritma

**Algoritma 1 — Protokol eksperimen keseluruhan**
```
Input : model M ∈ {XLM-R base, RoBERTa base}, metode {Full, LoRA r∈{4,8,16,32}}, strategi {S0,S1,S2}
1  Muat STS-B (train/val/test), SemRel2024 EN & ID, STS-B terjemahan ID; verifikasi pasangan EN–ID paralel
2  untuk setiap (M, metode, rank): jalankan Algoritma 3 pada S0, seed 42  → LR awal η0
3  untuk setiap (M, metode, rank, strategi): latih seed 42 dengan η0; bila collapse → turunkan LR satu langkah grid (Algoritma 3)
4  untuk setiap konfigurasi (30) dan seed k∈{1..5}: Algoritma 2  → bobot terbaik berdasarkan Spearman validasi
5  pilih Full terbaik dan LoRA terbaik per model (rerata Spearman validasi, laju divergensi ≤ 20 %)
6  tambah seed {6..10} untuk dua konfigurasi terpilih → 10 seed
7  ablasi (rsLoRA r∈{16,32}; q,v-saja r∈{8,16}) dan benchmark GPU
8  evaluasi SEKALI pada STS-B test (EN), SemRel EN, STS-B test (ID), SemRel ID
9  Algoritma 4 untuk RQ1, RQ2 (interaksi), RQ3 (Holm per famili)
Output: tabel, grafik, log sumber daya
```

**Algoritma 2 — Pelatihan SBERT (Full atau LoRA) dengan checkpoint**
```
Input : cfg (model, metode, rank, strategi, η, seed), data latih/validasi
1  id ← hash(cfg); bila result.json ada → lewati; bila checkpoint ada → lanjutkan dari epoch terakhir
2  inisialisasi encoder; bila LoRA: bekukan W0, pasang (A,B) pada semua lapisan linear
3  untuk epoch e = 1..4:
4      urutan data ← permutasi deterministik(seed·1000 + e)
5      untuk setiap batch (B=32): u,v ← SBERT(a), SBERT(b); L ← MSE (S0,S1) atau CoSENT (S2)
6          backprop; clip norma gradien 1,0; AdamW (LLRD pada S1); scheduler.step()
7      ρ_val ← Spearman(prediksi validasi); simpan bobot terlatih, optimizer, RNG, riwayat (atomik)
8  bobot terbaik ← epoch dengan ρ_val tertinggi; diverged ← (max ρ_val < 0,50)
9  catat waktu, memori puncak, daya/energi (NVML), ukuran checkpoint
```

**Algoritma 3 — Pencarian LR dengan deteksi collapse**
```
Input : grid G (urut menurun), konfigurasi (M, metode, rank, strategi)
1  untuk η ∈ G: latih (Algoritma 2, seed 42); simpan ρ_val dan flag diverged
2  η* ← argmax ρ_val di antara run yang TIDAK diverged
3  (saat screening strategi S1/S2) bila run dengan η* diverged: η* ← LR berikutnya yang lebih kecil di G; ulangi
4  bila G habis dan masih diverged → hentikan (jeda otomatis) dan laporkan
Output: LR per (model, metode, rank, strategi); catatan fallback
```

**Algoritma 4 — Perbandingan berpasangan LoRA vs Full**
```
Input : prediksi uji per seed, seed yang lolos (tidak diverged) pada kedua metode
1  d_k ← ρ_LoRA,k − ρ_Full,k untuk seed bersama (n ≥ 3)
2  hitung d̄, s_d, CI-t 95 %, p_TOST (δ = 0,01)
3  bootstrap kalimat uji (B_boot) pada rerata prediksi lintas seed → CI 95 % Δρ
4  (hanya bila n ≥ 10) uji Wilcoxon signed-rank
5  koreksi Holm dalam satu famili pengujian
6  keputusan: ekuivalen bila p_Holm < 0,05; "LoRA lebih buruk/baik" bila seluruh CI-t di luar [−δ, +δ]
```

## Referensi (APA)
- Cer, D., Diab, M., Agirre, E., Lopez-Gazpio, I., & Specia, L. (2017). SemEval-2017 Task 1: Semantic textual similarity multilingual and crosslingual focused evaluation. *Proceedings of SemEval-2017*, 1–14. https://doi.org/10.18653/v1/S17-2001
- Conneau, A., Khandelwal, K., Goyal, N., Chaudhary, V., Wenzek, G., Guzmán, F., Grave, E., Ott, M., Zettlemoyer, L., & Stoyanov, V. (2020). Unsupervised cross-lingual representation learning at scale. *Proceedings of ACL 2020*, 8440–8451. https://doi.org/10.18653/v1/2020.acl-main.747
- Dodge, J., Gururangan, S., Card, D., Schwartz, R., & Smith, N. A. (2019). Show your work: Improved reporting of experimental results. *Proceedings of EMNLP-IJCNLP 2019*, 2185–2194. https://doi.org/10.18653/v1/D19-1224
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics, 6*(2), 65–70. https://www.jstor.org/stable/4615733
- Howard, J., & Ruder, S. (2018). Universal language model fine-tuning for text classification. *Proceedings of ACL 2018*, 328–339. https://doi.org/10.18653/v1/P18-1031
- Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2022). LoRA: Low-rank adaptation of large language models. *ICLR 2022*. https://arxiv.org/abs/2106.09685
- Kalajdzievski, D. (2023). A rank stabilization scaling factor for fine-tuning with LoRA. *arXiv:2312.03732*. https://arxiv.org/abs/2312.03732
- Lakens, D. (2017). Equivalence tests: A practical primer for t tests, correlations, and meta-analyses. *Social Psychological and Personality Science, 8*(4), 355–362. https://doi.org/10.1177/1948550617697177
- Liu, Y., Ott, M., Goyal, N., Du, J., Joshi, M., Chen, D., Levy, O., Lewis, M., Zettlemoyer, L., & Stoyanov, V. (2019). RoBERTa: A robustly optimized BERT pretraining approach. *arXiv:1907.11692*. https://arxiv.org/abs/1907.11692
- Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR 2019*. https://arxiv.org/abs/1711.05101
- Ousidhoum, N., Muhammad, S. H., Abdalla, M., et al. (2024). SemRel2024: A collection of semantic textual relatedness datasets for 13 languages. *arXiv:2402.08638*. https://arxiv.org/abs/2402.08638
- Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. *Proceedings of EMNLP-IJCNLP 2019*, 3982–3992. https://doi.org/10.18653/v1/D19-1410
- Wolf, T., et al. (2020). Transformers: State-of-the-art natural language processing. *Proceedings of EMNLP 2020: System Demonstrations*, 38–45. https://doi.org/10.18653/v1/2020.emnlp-demos.6
- Mangrulkar, S., Gugger, S., Debut, L., Belkada, Y., Paul, S., & Bossan, B. (2022). *PEFT: State-of-the-art parameter-efficient fine-tuning methods* [Software]. Hugging Face. https://github.com/huggingface/peft
- Su, J. (2022). *CoSENT (一)：比Sentence-BERT更有效的句向量方案* [Blog post]. https://kexue.fm/archives/8847